# Avance Fase 3 — Semana 2

## Núcleo algorítmico, eficiencia e implementación orientada a objetos

**Proyecto:** Uso de redes sociales y salud mental percibida en adolescentes (YRBS 2023, CDC)
**Grupo 8 · MCDI500 Programación para la Ciencia de Datos** — Abigail Roblez Chavez, Daniel Pérez Ramírez, Matías Manríquez Ortiz, Roberto Sánchez Saldivia
**Docente:** Dr. Omar Salinas Silva

---

En la Fase 2 construimos un pipeline que funciona: lee el archivo original del CDC,
selecciona 11 de 117 columnas, conserva los faltantes con una bandera, declara el tipo de
cada código y crea columnas 0/1. Este cuaderno toma **ese mismo pipeline** y lo organiza
como un **sistema de clases**: cada paso pasa a ser una pieza que se puede probar,
reemplazar y reutilizar.

No hay datos nuevos ni pipeline nuevo. **Lo que cambia es la arquitectura del código.** La
prueba de que la reorganización no rompió nada está en la sección 14: el pipeline de clases
produce un archivo **idéntico, byte a byte**, al de la Fase 2.

> **Sobre los ejemplos de las secciones 2 a 10.** Para explicar cada concepto se usan clases
> pequeñas (imputar, escalar, codificar). Son **didácticas**: en el proyecto no se imputa ni
> se escala (bitácora 2.5 y sección 5 de F2). El pipeline real del proyecto empieza en la
> sección 11.

### Recorrido

| Parte | Contenido | Criterio de la rúbrica que alimenta |
|---|---|---|
| 0 | Configuración, carga y perfil del conjunto | Codificación funcional |
| 1 | De funciones a objetos: por qué | Codificación funcional |
| 2 | La clase `Preprocesador` | Programación orientada a objetos |
| 3 | Encapsulamiento: proteger el estado | Programación orientada a objetos |
| 4 | Herencia y polimorfismo | Programación orientada a objetos |
| 5 | El `Pipeline`: componer los pasos | Diseño estructurado |
| 6 | Cohesión y acoplamiento | Diseño estructurado |
| 7 | Validación de las clases base | Validación técnica |
| 8 | Recursividad | Diseño estructurado |
| 9 | Eficiencia: medir tiempo y memoria | Eficiencia y optimización |
| 10 | Patrones de diseño (Strategy, Factory, Observer) | Documentación de arquitectura |
| **11** | **El pipeline de la Fase 2 en clases** (ejercicios 1 y 2) | POO · Preprocesamiento · Diseño |
| **12** | **Eficiencia sobre un paso real** (ejercicio 3) | Eficiencia y optimización |
| **13** | **Validación del pipeline real** (ejercicio 4) | Validación técnica |
| **14** | **Strategy y verificación contra la Fase 2** (ejercicio 5) | Arquitectura · Validación |
| 15 | Resultado, arquitectura y registro | Documentación de arquitectura |

**Cómo ejecutarlo.** Abrir desde `F3/notebooks/` y usar *Restart Kernel and Run All Cells*.
Los datos se leen desde fuera: el archivo original en `F1/data/raw/` y el procesado de la
Fase 2 en `F2/data/processed/`.

---
## Configuración

**Esta es la única celda que hay que editar.** Todo lo que sigue se construye a partir
de ella.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# Todas las rutas y nombres de columna se declaran acá, una sola vez.
# Las rutas se escriben desde la carpeta de este cuaderno (F3/notebooks).
# =====================================================================
from pathlib import Path

RAIZ = Path("..") / ".."
RUTA_DATOS = RAIZ / "F2" / "data" / "processed" / "yrbs2023_seleccion_procesada.csv"  # salida de F2
RUTA_ORIGINAL = RAIZ / "F1" / "data" / "raw" / "XXH2023_YRBSS_data.csv"              # archivo del CDC
RUTA_SRC = RAIZ / "src"                                                           # módulo de F2
CARPETA_RESULTADOS = RAIZ / "F3" / "resultados"                                   # salidas de F3

COLUMNA_OBJETIVO = "salud_mental_cod"          # desenlace: salud mental percibida (q84)
COLUMNA_ID = "id_registro"                     # identificador: no describe, solo identifica

COLUMNAS_CONTINUAS = ["redes_sociales_cod", "sueno_cod", "actividad_fisica_cod", "edad_cod"]
COLUMNAS_NOMINALES = ["raceeth_cod", "sexo_cod"]

SEMILLA = 42
PROPORCION_PRUEBA = 0.2

# Columnas usadas en los ejemplos didácticos (secciones 2 a 10)
COLUMNA_IMPUTAR = "actividad_fisica_cod"   # con faltantes (6,1 %): ejemplos de imputación
COLUMNA_ESCALAR = "edad_cod"               # ejemplos de escalamiento
COLUMNA_GRUPO = "sexo_cod"                 # imputación por grupo (Strategy didáctico)
COLUMNA_CLASIFICAR = "sueno_cod"           # comparación bucle vs. vectorizado
# =====================================================================

COLUMNAS_ESPERADAS = ([COLUMNA_ID, COLUMNA_OBJETIVO] + COLUMNAS_CONTINUAS
                      + COLUMNAS_NOMINALES)

print("Archivo de la Fase 2:", RUTA_DATOS)
print("Archivo original    :", RUTA_ORIGINAL)
print("Variable objetivo   :", COLUMNA_OBJETIVO)
print("Columnas esperadas  :", len(COLUMNAS_ESPERADAS))

Archivo de la Fase 2: ..\..\F2\data\processed\yrbs2023_seleccion_procesada.csv
Archivo original    : ..\..\F1\data\raw\XXH2023_YRBSS_data.csv
Variable objetivo   : salud_mental_cod
Columnas esperadas  : 8


## Preparación del entorno

In [2]:
import os
import io
import sys
import time
import timeit
import tracemalloc

import numpy as np
import pandas as pd

np.random.seed(SEMILLA)

# El módulo de la Fase 2 (src/procesamiento.py) se reutiliza: sus constantes
# (columnas, escalas del codebook) son la fuente única de verdad del proyecto.
if str(RUTA_SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(RUTA_SRC.resolve()))
import procesamiento as proc

print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)
print("Módulo de F2 cargado:", Path(proc.__file__).name)

Python : 3.14.7
pandas : 3.0.6
NumPy  : 2.5.3
Semilla: 42
Módulo de F2 cargado: procesamiento.py


## Carga del conjunto desde el archivo externo

El cuaderno **no genera los datos**: los lee del archivo que se indique en la
configuración. La función de carga hace tres cosas antes de devolver nada:

1. Comprueba que el archivo exista, y si no, explica dónde ponerlo.
2. Lo lee según su extensión, sea CSV o Excel.
3. Verifica que estén las columnas declaradas en la configuración.

Ese tercer paso es el que evita el error más molesto: descubrir en la celda veinte que
una columna se llamaba distinto.

In [3]:
def leer_archivo(ruta):
    """Lee el archivo según su extensión y devuelve un DataFrame."""
    extension = os.path.splitext(ruta)[1].lower()

    if extension in (".csv", ".txt"):
        # sep=None con engine="python" infiere el separador: útil con datos
        # públicos chilenos, que suelen venir con punto y coma
        return pd.read_csv(ruta, sep=None, engine="python", encoding="utf-8")
    if extension in (".xlsx", ".xls"):
        return pd.read_excel(ruta)
    if extension == ".parquet":
        return pd.read_parquet(ruta)

    raise ValueError(
        f"Extensión no reconocida: '{extension}'. "
        "Se admiten .csv, .txt, .xlsx, .xls y .parquet."
    )


def verificar_esquema(df, columnas_esperadas):
    """Comprueba que estén todas las columnas declaradas en la configuración."""
    faltantes = [c for c in columnas_esperadas if c not in df.columns]
    if faltantes:
        raise KeyError(
            f"Faltan columnas declaradas en la configuración: {faltantes}\n"
            f"Columnas disponibles en el archivo: {list(df.columns)}"
        )
    sobrantes = [c for c in df.columns if c not in columnas_esperadas]
    return {"declaradas": len(columnas_esperadas),
            "en_archivo": df.shape[1],
            "no_declaradas": sobrantes}

In [4]:
def cargar(ruta=RUTA_DATOS, columnas_esperadas=None):
    """Carga el conjunto desde el archivo externo y verifica su esquema."""
    columnas_esperadas = columnas_esperadas or COLUMNAS_ESPERADAS

    if not os.path.exists(ruta):
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}\n"
            f"Carpeta actual: {os.getcwd()}\n"
            "Revisen RUTA_DATOS en la celda de configuración. "
            "La ruta se escribe desde la carpeta donde está este cuaderno."
        )
    df = leer_archivo(ruta)
    print(f"Archivo leído: {ruta}")

    # Las columnas de códigos se fuerzan a número: un texto suelto las
    # convertiría en columna de objetos sin ningún aviso
    for columna in COLUMNAS_CONTINUAS:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")

    informe = verificar_esquema(df, columnas_esperadas)
    print(f"Forma: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"Esquema verificado: {informe['declaradas']} columnas declaradas, "
          f"{len(informe['no_declaradas'])} no declaradas")
    return df


datos = cargar()
datos.head()

Archivo leído: ..\..\F2\data\processed\yrbs2023_seleccion_procesada.csv
Forma: 20103 filas x 26 columnas
Esquema verificado: 8 columnas declaradas, 18 no declaradas


,id_registro,peso_muestral,estrato,psu,edad_cod,sexo_cod,raceeth_cod,salud_mental_cod,redes_sociales_cod,sueno_cod,...,raza_hawaiana_pacifico,raza_blanca,raza_hispana,raza_multiple_hispana,raza_multiple_no_hispana,sexo_femenino,salud_mental_mala,redes_uso_frecuente,sueno_8h_o_mas,actividad_5_dias
0,1,0.8614,103,16294,3.0,1.0,NaN,1.0,6.0,3.0,...,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1.0,0.0,0.0
1,2,0.8920,103,16294,4.0,2.0,5.0,3.0,4.0,5.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,3,0.5081,103,16294,5.0,2.0,5.0,2.0,8.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,4,1.1759,103,16294,6.0,1.0,5.0,3.0,8.0,4.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,5,0.8920,103,16294,3.0,2.0,5.0,3.0,6.0,3.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


**Si se obtiene un `KeyError` al cargar**, la configuración no coincide con el archivo. El
mensaje lista las columnas que faltan y las que sí están: corrijan los nombres en la celda
de configuración y vuelvan a ejecutar.

**Las 26 columnas** son las 11 seleccionadas en F2 más 15 derivadas (bandera de faltantes,
8 columnas de raza/etnicidad y 5 indicadores 0/1). Las «no declaradas» son esas derivadas.

## El conjunto antes de tocarlo

Un perfil que funciona con cualquier conjunto, no solo con este. Se apoya en la
configuración, así que al cambiar de proyecto sigue sirviendo.

In [5]:
def perfilar(df):
    """Devuelve una fila por columna con su tipo, únicos y porcentaje de nulos."""
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).sort_values("pct_nulos", ascending=False).reset_index(drop=True)


perfil = perfilar(datos)
print("Columnas con valores faltantes:")
print(perfil[perfil["nulos"] > 0].to_string(index=False))

print(f"\nDistribución de la variable objetivo '{COLUMNA_OBJETIVO}':")
print(datos[COLUMNA_OBJETIVO].value_counts(normalize=True).round(4))

for columna in COLUMNAS_NOMINALES:
    print(f"\nCategorías de {columna}:")
    print(datos[columna].value_counts(dropna=False).sort_index())

Columnas con valores faltantes:
                 columna    tipo  unicos  nulos  pct_nulos
      redes_sociales_cod float64       8   4900      24.37
     redes_uso_frecuente float64       2   4900      24.37
       salud_mental_mala float64       2   4398      21.88
        salud_mental_cod float64       5   4398      21.88
               sueno_cod float64       7   2662      13.24
          sueno_8h_o_mas float64       2   2662      13.24
    actividad_fisica_cod float64       8   1227       6.10
        actividad_5_dias float64       2   1227       6.10
          raza_amerindia float64       2    370       1.84
             raceeth_cod float64       8    370       1.84
   raza_multiple_hispana float64       2    370       1.84
raza_multiple_no_hispana float64       2    370       1.84
             raza_blanca float64       2    370       1.84
  raza_hawaiana_pacifico float64       2    370       1.84
            raza_hispana float64       2    370       1.84
           raza_asiatica

**Tres observaciones que condicionan todo lo que viene (YRBS 2023).**

1. **Hay faltantes reales en todas las variables de análisis**, entre 0,5 % (`edad_cod`) y 24,4 %
   (`redes_sociales_cod`). En la Fase 2 se mostró que no son aleatorios y se decidió **no imputarlos**
   (bitácora 2.5). Aquí la imputación aparece solo como ejemplo didáctico sobre `COLUMNA_IMPUTAR`.
2. **Los códigos son ordinales o nominales, no cantidades.** `sexo_cod` y `raceeth_cod` son nominales:
   un código 6 no «vale más» que un 2. Por eso se codifican con columnas 0/1.
3. **La variable objetivo (`salud_mental_cod`) tiene 21,9 % de faltantes.** No se imputa: la separación
   entrenamiento/prueba se hace solo con los registros que la respondieron (sección 3).

---
# 1. De funciones a objetos: por qué

Así se ve el código de la Fase 2, en celdas sueltas:

In [6]:
# El estilo de la Fase 2: funciona, pero cada celda depende de las anteriores
df_f2 = datos.copy()
df_f2 = df_f2.drop(columns=[COLUMNA_ID])
df_f2[COLUMNA_IMPUTAR] = df_f2[COLUMNA_IMPUTAR].fillna(df_f2[COLUMNA_IMPUTAR].median())
df_f2 = pd.get_dummies(df_f2, columns=["raceeth_cod"], dtype=int)

print("Resultado:", df_f2.shape)

Resultado: (20103, 32)


Funciona. Pero tiene cuatro problemas que aparecen cuando el proyecto crece:

| Problema | Consecuencia |
|---|---|
| El orden importa y no está declarado | Ejecutar una celda fuera de orden produce otro resultado |
| No se puede reutilizar sin copiar | El mismo código se duplica en el cuaderno de F3 y F4 |
| No se puede probar por partes | Si el resultado está mal, hay que revisar todo |
| La mediana se calcula sobre todo el conjunto | Si después separan entrenamiento y prueba, hay **fuga de datos** |

La programación orientada a objetos resuelve los cuatro: cada paso pasa a ser una pieza
con nombre, con estado propio y con una interfaz fija.

---
# 2. La clase `Preprocesador`

Primera versión: una clase que guarda la tabla y sabe prepararla. Cada método hace
**una sola cosa** y devuelve `self`, lo que permite encadenar llamadas.

In [7]:
class Preprocesador:
    """Guarda la tabla del proyecto y sabe prepararla para el análisis.

    La tabla vive en self.df. Cada método es un paso del pipeline que trabaja
    sobre esa misma tabla y devuelve self, para poder encadenar.
    """

    def __init__(self, df):
        self.df = df.copy()          # copia: no se modifica el original
        self.registro = []           # lo que el objeto va recordando

    def quitar_identificador(self):
        """El id no aporta información al análisis: identifica, no describe."""
        if COLUMNA_ID in self.df.columns:
            self.df = self.df.drop(columns=[COLUMNA_ID])
            self.registro.append(f"{COLUMNA_ID} eliminado")
        return self

    def imputar(self, columna):
        """Rellena los nulos de una columna con la mediana."""
        nulos = int(self.df[columna].isna().sum())
        mediana = float(self.df[columna].median())
        self.df[columna] = self.df[columna].fillna(mediana)
        self.registro.append(f"{columna}: {nulos} nulos imputados con mediana={mediana:.1f}")
        return self

    def codificar(self, columnas):
        """Convierte columnas de categorías en columnas 0/1."""
        antes = self.df.shape[1]
        self.df = pd.get_dummies(self.df, columns=columnas, dtype=int)
        self.registro.append(f"codificadas {columnas}: {antes} -> {self.df.shape[1]} columnas")
        return self

    def informe(self):
        """Devuelve lo que el objeto recuerda haber hecho."""
        return "\n".join(f"  {i}. {paso}" for i, paso in enumerate(self.registro, 1))


prep = Preprocesador(datos)
prep.quitar_identificador().imputar(COLUMNA_IMPUTAR).codificar(COLUMNAS_NOMINALES)

print("Resultado:", prep.df.shape)
print("\nLo que el objeto recuerda:")
print(prep.informe())

Resultado: (20103, 33)

Lo que el objeto recuerda:
  1. id_registro eliminado
  2. actividad_fisica_cod: 1227 nulos imputados con mediana=5.0
  3. codificadas ['raceeth_cod', 'sexo_cod']: 25 -> 33 columnas


**Lo que ganamos con esto.** El objeto `prep` recuerda qué hizo y con qué valores. En la
Fase 2 esa información había que anotarla a mano en la bitácora; acá es producto de la
ejecución y no puede quedar desactualizada.

**Lo que todavía falta.** Esta clase mezcla dos cosas: aprender de los datos y
aplicarlos. Eso se arregla en la parte siguiente.

---
# 3. Encapsulamiento: proteger el estado interno

El `Preprocesador` de arriba tiene un problema serio: calcula la mediana **sobre el
conjunto completo**. Si después separan entrenamiento y prueba, la mediana ya vio los
datos de prueba. Eso se llama **fuga de datos** y es uno de los errores más caros en
ciencia de datos.

La solución es separar dos momentos:

- **`ajustar`**: aprende los parámetros, y solo del conjunto de entrenamiento.
- **`transformar`**: los aplica a cualquier conjunto.

Y proteger ese estado interno con un guion bajo al inicio del nombre, que en Python
significa «esto es interno, no lo toques desde fuera».

In [8]:
class ImputadorSimple:
    """Imputa una columna separando lo que aprende de lo que aplica (ejemplo didáctico)."""

    def __init__(self, columna=COLUMNA_IMPUTAR):
        self.columna = columna
        self._mediana = None        # interno: se aprende, no se asigna desde fuera
        self._ajustado = False      # interno: controla el orden de las llamadas

    @property
    def ajustado(self):
        """Solo lectura: se puede consultar, no asignar."""
        return self._ajustado

    @property
    def mediana(self):
        return self._mediana

    def ajustar(self, df):
        self._mediana = float(df[self.columna].median())
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError(
                "ImputadorSimple: hay que llamar a ajustar() antes de transformar(). "
                "La mediana se aprende del conjunto de entrenamiento."
            )
        df = df.copy()
        df[self.columna] = df[self.columna].fillna(self._mediana)
        return df


# El orden equivocado ahora falla con un mensaje claro
imputador = ImputadorSimple()
try:
    imputador.transformar(datos)
except RuntimeError as error:
    print("RuntimeError:", error)

RuntimeError: ImputadorSimple: hay que llamar a ajustar() antes de transformar(). La mediana se aprende del conjunto de entrenamiento.


In [9]:
# El orden correcto, con separación de conjuntos
# La variable objetivo no se imputa (bitácora 2.5): solo se separan los registros
# que la respondieron
datos_objetivo = datos.dropna(subset=[COLUMNA_OBJETIVO])
print(f"Registros con {COLUMNA_OBJETIVO} respondida: {len(datos_objetivo)} de {len(datos)}")

entrenamiento = datos_objetivo.sample(frac=1 - PROPORCION_PRUEBA, random_state=SEMILLA)
prueba = datos_objetivo.drop(index=entrenamiento.index)

imputador = ImputadorSimple().ajustar(entrenamiento)
prueba_lista = imputador.transformar(prueba)

print("Mediana aprendida del entrenamiento:", round(imputador.mediana, 2))
print("Mediana del conjunto de prueba     :", round(prueba[COLUMNA_IMPUTAR].median(), 2))
print("Nulos en prueba después de imputar :", int(prueba_lista[COLUMNA_IMPUTAR].isna().sum()))
if imputador.mediana != prueba[COLUMNA_IMPUTAR].median():
    print("\nSon medianas distintas, y está bien: se usó la del entrenamiento.")
else:
    print("\nAquí coinciden porque el código es discreto (1 a 8), pero el valor usado es el del entrenamiento.")

# La propiedad es de solo lectura
try:
    imputador.ajustado = False
except AttributeError as error:
    print("\nAttributeError:", error)

Registros con salud_mental_cod respondida: 15705 de 20103
Mediana aprendida del entrenamiento: 5.0
Mediana del conjunto de prueba     : 5.0
Nulos en prueba después de imputar : 0

Aquí coinciden porque el código es discreto (1 a 8), pero el valor usado es el del entrenamiento.

AttributeError: property 'ajustado' of 'ImputadorSimple' object has no setter


**Por qué importa.** Sin el control de `_ajustado`, transformar antes de ajustar
produciría un resultado sin sentido **y sin ningún mensaje de error**. El
encapsulamiento convierte un error silencioso en uno visible.

En el informe, este bloque es la evidencia del encapsulamiento: atributos internos con
guion bajo, propiedades de solo lectura y validación antes de actuar.

---
# 4. Herencia y polimorfismo

Al mirar el `ImputadorSimple`: el control de `_ajustado`, la copia defensiva y el mensaje de
error van a repetirse en el codificador, en el escalador y en cualquier otro paso.

**Herencia** es escribir eso una sola vez en una clase base.

In [10]:
class Transformador:
    """Clase base: lo que todos los pasos del pipeline comparten.

    Esta clase NO se usa directamente. Su trabajo es definir el contrato:
    qué métodos tendrá todo paso del pipeline y cómo se controla su estado.
    Las clases hijas solo completan lo que cambia de un paso a otro.
    """

    def __init__(self, columna):
        # Atributo público: cualquiera puede leerlo y cambiarlo
        self.columna = columna

        # Atributos con guion bajo: por convención, INTERNOS.
        # Python no lo impide, pero el guion bajo le dice a quien lee el código
        # que no debe tocarlos desde fuera de la clase.
        self._parametros = {}       # lo que el paso aprende del conjunto
        self._ajustado = False      # controla que no se transforme antes de ajustar

    # @property convierte un método en algo que se lee como atributo:
    # se escribe paso.nombre y no paso.nombre(). Y como no hay setter,
    # el valor NO se puede asignar desde fuera. Eso es encapsulamiento.
    @property
    def nombre(self):
        # type(self).__name__ devuelve el nombre de la clase REAL del objeto,
        # así que un ImputadorMediana se identifica como tal y no como Transformador
        return f"{type(self).__name__}({self.columna})"

    @property
    def parametros(self):
        # dict(...) crea una COPIA. Si devolviéramos self._parametros
        # directamente, quien lo recibe podría modificar el estado interno
        # del objeto sin que la clase se entere. A esto se le llama
        # copia defensiva.
        return dict(self._parametros)

    def columnas_requeridas(self):
        """Columnas que el paso necesita encontrar. Por defecto, solo la suya.

        Las clases hijas que trabajan sobre varias columnas lo redefinen
        (sección 11). Es un ejemplo pequeño de polimorfismo.
        """
        return [self.columna]

    def ajustar(self, df):
        """Aprende los parámetros del conjunto que recibe."""
        # 1) Validar la entrada ANTES de trabajar: si algo falta, se avisa
        #    acá y no veinte líneas más abajo con un error incomprensible
        faltan = [c for c in self.columnas_requeridas() if c not in df.columns]
        if faltan:
            raise KeyError(f"{self.nombre}: no existen las columnas {faltan}.")

        # 2) Delegar el cálculo a la clase hija. La base no sabe QUÉ se aprende;
        #    solo sabe CUÁNDO hay que aprenderlo.
        self._parametros = self.aprender(df)

        # 3) Registrar que ya se ajustó, para poder controlarlo después
        self._ajustado = True

        # 4) Devolver self permite encadenar: paso.ajustar(df).transformar(df)
        return self

    def transformar(self, df):
        """Aplica la transformación usando lo aprendido en ajustar()."""
        # Esta guarda es el corazón del encapsulamiento: impide un uso
        # incorrecto que de otro modo pasaría inadvertido
        if not self._ajustado:
            raise RuntimeError(f"{self.nombre}: hay que ajustar antes de transformar.")

        # df.copy() evita modificar el DataFrame que nos pasaron. Sin esto,
        # el objeto original del usuario cambiaría sin que él lo pidiera.
        return self.aplicar(df.copy())

    def ajustar_transformar(self, df):
        """Atajo para el caso habitual: aprender y aplicar sobre el mismo conjunto."""
        return self.ajustar(df).transformar(df)

    # ---- Métodos que cada clase hija DEBE implementar -------------------
    # Se definen acá lanzando NotImplementedError para dejar el contrato
    # explícito: si una hija olvida implementarlos, el error lo dice claro.

    def aprender(self, df):
        """Calcula y devuelve los parámetros. Lo implementa cada clase hija."""
        raise NotImplementedError("Cada clase hija debe implementar aprender().")

    def aplicar(self, df):
        """Aplica la transformación. Lo implementa cada clase hija."""
        raise NotImplementedError("Cada clase hija debe implementar aplicar().")


# La clase base no se usa sola
try:
    Transformador(COLUMNA_IMPUTAR).ajustar(datos)
except NotImplementedError as error:
    print("NotImplementedError:", error)

NotImplementedError: Cada clase hija debe implementar aprender().


In [11]:
class ImputadorMediana(Transformador):
    """Rellena los nulos con la mediana aprendida.

    El paréntesis (Transformador) es la HERENCIA: esta clase recibe todo lo
    que tiene Transformador sin volver a escribirlo. Solo implementa los dos
    métodos que le faltaban.
    """

    def aprender(self, df):
        return {"mediana": float(df[self.columna].median()),
                "nulos_en_ajuste": int(df[self.columna].isna().sum())}

    def aplicar(self, df):
        df[self.columna] = df[self.columna].fillna(self._parametros["mediana"])
        return df


class CodificadorNominal(Transformador):
    """Convierte una columna de categorías en columnas 0/1.

    El vocabulario se aprende en el ajuste: una categoría que solo aparece en
    prueba no genera columna nueva, porque el modelo no pudo aprender de ella.
    """

    def aprender(self, df):
        # Se guarda la lista de categorías ORDENADA, para que el resultado sea
        # el mismo en cada ejecución. Sin sorted(), el orden podría variar.
        return {"categorias": sorted(df[self.columna].dropna().unique())}

    def aplicar(self, df):
        # Se recorre el vocabulario aprendido, NO las categorías de este df.
        # Por eso una categoría nueva en prueba no genera columna: el modelo
        # nunca la vio y no podría haber aprendido nada de ella.
        for categoria in self._parametros["categorias"]:
            # Los nombres de columna no deben llevar espacios ni guiones
            etiqueta = str(categoria).strip().replace(" ", "_").replace("-", "_")
            # La comparación devuelve True/False; astype(int) lo pasa a 1/0
            df[f"{self.columna}_{etiqueta}"] = (df[self.columna] == categoria).astype(int)

        # La columna original ya no aporta: su información quedó en las nuevas
        return df.drop(columns=[self.columna])


class EscaladorEstandar(Transformador):
    """Centra en cero y escala a desviación uno."""

    def aprender(self, df):
        desviacion = float(df[self.columna].std())
        return {"media": float(df[self.columna].mean()),
                "desviacion": desviacion if desviacion != 0 else 1.0}

    def aplicar(self, df):
        df[self.columna] = ((df[self.columna] - self._parametros["media"])
                            / self._parametros["desviacion"])
        return df


class EliminadorColumnas(Transformador):
    """Quita columnas que no aportan al análisis, como el identificador."""

    def aprender(self, df):
        return {"existe": self.columna in df.columns}

    def aplicar(self, df):
        return df.drop(columns=[self.columna]) if self.columna in df.columns else df


imp = ImputadorMediana(COLUMNA_IMPUTAR)
salida = imp.ajustar_transformar(datos)
print(imp.nombre, "->", imp.parametros)
print(f"Nulos en {COLUMNA_IMPUTAR} después:", int(salida[COLUMNA_IMPUTAR].isna().sum()))

ImputadorMediana(actividad_fisica_cod) -> {'mediana': 5.0, 'nulos_en_ajuste': 1227}
Nulos en actividad_fisica_cod después: 0


### Cómo comprobar que la herencia funciona

No hay que creer en la herencia: se puede verificar.

In [12]:
imputador = ImputadorMediana(COLUMNA_IMPUTAR)

# 1) El objeto ES un ImputadorMediana Y TAMBIÉN es un Transformador
print("¿Es ImputadorMediana?", isinstance(imputador, ImputadorMediana))
print("¿Es Transformador?   ", isinstance(imputador, Transformador))

# 2) La cadena de búsqueda muestra de dónde hereda
print("\nCadena de herencia:", [c.__name__ for c in ImputadorMediana.__mro__])

# 3) Métodos que NO están escritos en la clase hija y sin embargo funcionan,
#    porque se heredaron de la clase base
print("\nMétodos heredados de Transformador:")
for metodo in ["ajustar", "transformar", "ajustar_transformar"]:
    definido_en = "ImputadorMediana" if metodo in ImputadorMediana.__dict__ else "Transformador"
    print(f"  {metodo:<22} definido en {definido_en}")

print("\nMétodos propios de la clase hija:")
for metodo in ["aprender", "aplicar"]:
    print(f"  {metodo:<22} definido en ImputadorMediana")

¿Es ImputadorMediana? True
¿Es Transformador?    True

Cadena de herencia: ['ImputadorMediana', 'Transformador', 'object']

Métodos heredados de Transformador:
  ajustar                definido en Transformador
  transformar            definido en Transformador
  ajustar_transformar    definido en Transformador

Métodos propios de la clase hija:
  aprender               definido en ImputadorMediana
  aplicar                definido en ImputadorMediana


**Lo que acaban de ver.** `ImputadorMediana` tiene ocho líneas de código propio y
dispone de toda la maquinaria de control de estado. Eso es lo que la herencia ahorra: si
mañana hay que cambiar el mensaje de error, se cambia en un solo lugar y las cuatro
clases hijas lo heredan.

**Polimorfismo** es que quien usa estas clases no necesita saber de qué tipo es cada
una: todas ofrecen `ajustar` y `transformar`.

In [13]:
pasos = [
    EliminadorColumnas(COLUMNA_ID),
    ImputadorMediana(COLUMNA_IMPUTAR),
    CodificadorNominal("raceeth_cod"),
    CodificadorNominal("sexo_cod"),
    EscaladorEstandar(COLUMNA_ESCALAR),
]

resultado = datos.copy()
for paso in pasos:                      # la misma llamada para los cinco
    resultado = paso.ajustar_transformar(resultado)
    print(f"{paso.nombre:<34} -> {resultado.shape}")

EliminadorColumnas(id_registro)    -> (20103, 25)
ImputadorMediana(actividad_fisica_cod) -> (20103, 25)
CodificadorNominal(raceeth_cod)    -> (20103, 32)
CodificadorNominal(sexo_cod)       -> (20103, 33)
EscaladorEstandar(edad_cod)        -> (20103, 33)


**Este bloque es la evidencia de los tres principios**, y así conviene declararlo en el
informe:

- **Herencia**: las cuatro clases heredan de `Transformador` y no repiten el control de estado.
- **Polimorfismo**: el bucle llama `ajustar_transformar()` sin preguntar el tipo.
- **Encapsulamiento**: `_parametros` y `_ajustado` son internos, y hay validación antes de actuar.

Al fijarnos en la consecuencia práctica: **agregar un transformador nuevo no obliga a
tocar el bucle**. Eso es lo que la rúbrica llama bajo acoplamiento.

---
# 5. El `Pipeline`: componer los pasos

Una clase que guarda el orden, los ejecuta y controla que se haya ajustado antes de
transformar.

In [14]:
class Pipeline:
    """Encadena transformadores y los ejecuta en orden."""

    def __init__(self, pasos=None):
        self._pasos = list(pasos) if pasos else []
        self._ajustado = False

    def agregar(self, transformador):
        if not isinstance(transformador, Transformador):
            raise TypeError(
                f"Se esperaba un Transformador y se recibió {type(transformador).__name__}."
            )
        self._pasos.append(transformador)
        return self

    def ajustar(self, df):
        """Aprende los parámetros de cada paso SOLO con estos datos."""
        intermedio = df.copy()
        for paso in self._pasos:
            intermedio = paso.ajustar_transformar(intermedio)
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError("Pipeline: hay que ajustar antes de transformar.")
        resultado = df.copy()
        for paso in self._pasos:
            resultado = paso.transformar(resultado)
        return resultado

    def pasos_ejecutados(self):
        """Devuelve los pasos, para poder registrar sus parámetros aprendidos."""
        return tuple(self._pasos)

    def resumen(self):
        return pd.DataFrame([
            {"orden": i, "paso": p.nombre, "clase": type(p).__name__}
            for i, p in enumerate(self._pasos, start=1)
        ])

    def __len__(self):
        return len(self._pasos)

    def __repr__(self):
        estado = "ajustado" if self._ajustado else "sin ajustar"
        return f"Pipeline({len(self._pasos)} pasos, {estado})"


pipeline = Pipeline([
    EliminadorColumnas(COLUMNA_ID),
    ImputadorMediana(COLUMNA_IMPUTAR),
    CodificadorNominal("raceeth_cod"),
    CodificadorNominal("sexo_cod"),
    EscaladorEstandar(COLUMNA_ESCALAR),
    EscaladorEstandar(COLUMNA_IMPUTAR),
])

print(pipeline)
pipeline.resumen()

Pipeline(6 pasos, sin ajustar)


,orden,paso,clase
0,1,EliminadorColumnas(id_registro),EliminadorColumnas
1,2,ImputadorMediana(actividad_fisica_cod),ImputadorMediana
2,3,CodificadorNominal(raceeth_cod),CodificadorNominal
3,4,CodificadorNominal(sexo_cod),CodificadorNominal
4,5,EscaladorEstandar(edad_cod),EscaladorEstandar
5,6,EscaladorEstandar(actividad_fisica_cod),EscaladorEstandar


In [15]:
# Ajustar con entrenamiento, transformar prueba: sin fuga de datos
pipeline.ajustar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, " -> Prueba transformada:", prueba_lista.shape)
print(f"\nMedia de {COLUMNA_ESCALAR} en prueba tras escalar:", round(prueba_lista[COLUMNA_ESCALAR].mean(), 4))
print("No es exactamente cero, y está bien: los parámetros vienen del entrenamiento.")

Entrenamiento: (12564, 26)  -> Prueba transformada: (3141, 33)

Media de edad_cod en prueba tras escalar: -0.013
No es exactamente cero, y está bien: los parámetros vienen del entrenamiento.


**Por qué la media no da cero.** El escalador aprendió la media del entrenamiento. Si
diera exactamente cero sobre prueba, significaría que aprendió de datos que no debía
ver. Este resultado es la confirmación de que el diseño está bien.

---
# 6. Cohesión y acoplamiento

Son las dos palabras que más aparecen en el criterio de diseño estructurado y que casi
ningún material explica. Acá van con código.

**Alta cohesión** significa que cada componente hace **una sola cosa**.

**Bajo acoplamiento** significa que los componentes dependen poco unos de otros: cambiar
uno no obliga a tocar los demás.

## 6.1 Cohesión baja: una clase que hace de todo

Este es el diseño que aparece cuando se escribe rápido. Funciona, pero cada método toca
cosas distintas y la clase no tiene un propósito único.

In [16]:
class ProcesadorTodoEnUno:
    """Ejemplo de COHESIÓN BAJA: hace cuatro cosas sin relación entre sí."""

    def __init__(self, ruta):
        self.ruta = ruta
        self.df = None

    def cargar(self):            # responsabilidad 1: entrada y salida de archivos
        self.df = pd.read_csv(self.ruta)

    def limpiar(self):           # responsabilidad 2: transformar datos
        self.df = self.df.dropna()

    def graficar(self):          # responsabilidad 3: visualización
        pass

    def enviar_correo(self):     # responsabilidad 4: comunicación
        pass


print("Cuatro responsabilidades distintas en una sola clase.")
print("Problema práctico: para probar la limpieza hay que tener un archivo real,")
print("porque cargar() y limpiar() viven pegadas en el mismo objeto.")

Cuatro responsabilidades distintas en una sola clase.
Problema práctico: para probar la limpieza hay que tener un archivo real,
porque cargar() y limpiar() viven pegadas en el mismo objeto.


## 6.2 Cohesión alta: cada clase con un propósito

El mismo trabajo, repartido. Ahora cada pieza se puede probar sola.

In [17]:
class Cargador:
    """Una sola responsabilidad: leer datos desde una fuente."""

    def __init__(self, ruta):
        self.ruta = ruta

    def cargar(self):
        return pd.read_csv(self.ruta)


class Limpiador:
    """Una sola responsabilidad: transformar un DataFrame que le entregan.

    Fíjense en que NO sabe de dónde vienen los datos. Recibe un DataFrame y
    devuelve otro. Eso es bajo acoplamiento: no depende del Cargador.
    """

    def limpiar(self, df):
        return df.dropna()


# La ventaja se ve al probar: no hace falta ningún archivo
mini = pd.DataFrame({"a": [1.0, np.nan, 3.0], "b": [4.0, 5.0, 6.0]})
print("Se puede probar el Limpiador sin tocar el disco:")
print("  entrada:", mini.shape, " -> salida:", Limpiador().limpiar(mini).shape)

Se puede probar el Limpiador sin tocar el disco:
  entrada: (3, 2)  -> salida: (2, 2)


## 6.3 La prueba del acoplamiento

Hay una pregunta que resuelve el asunto sin teoría:

> **Si cambiamos la forma de imputar, ¿cuántos archivos hay que modificar?**

Si la respuesta es **uno**, el acoplamiento es bajo. Si son tres, el diseño es frágil.

Veámoslo con el pipeline que ya construimos.

In [18]:
# Cambiar un paso del pipeline no obliga a tocar ni el Pipeline ni los otros pasos
pipeline_a = Pipeline([EliminadorColumnas(COLUMNA_ID), ImputadorMediana(COLUMNA_IMPUTAR)])
pipeline_b = Pipeline([EliminadorColumnas(COLUMNA_ID), ImputadorMediana("sueno_cod")])

for nombre, pipe in [("versión A", pipeline_a), ("versión B", pipeline_b)]:
    salida = pipe.ajustar(datos).transformar(datos)
    print(f"{nombre}: {salida.shape}")

print("\nSe cambió el paso y NO se modificó la clase Pipeline ni las demás clases.")
print("Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.")

versión A: (20103, 25)
versión B: (20103, 25)

Se cambió el paso y NO se modificó la clase Pipeline ni las demás clases.
Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.


## 6.4 Cómo se traduce en la estructura de archivos

Una organización con alta cohesión y bajo acoplamiento se reconoce a simple vista. Es la que
proponemos para el repositorio (sección 15.2):

```
F3/src/
├── transformadores.py  la clase base Transformador y sus hijas
├── estrategias.py      las estrategias de faltantes (Strategy)
├── pipeline.py         Pipeline y PipelineObservable
└── medicion.py         medir() y las comparaciones de eficiencia
```

Cada archivo tiene un propósito que se puede enunciar en una frase.

**La prueba del acoplamiento, aplicada al proyecto:** para cambiar cómo se tratan los
faltantes solo se cambia la estrategia que recibe `TratamientoFaltantes` (sección 14); el
`Pipeline` y los demás pasos no se tocan.

---
# 7. Validación de las clases base: casos normales, límite y excepciones

Primero se validan las clases didácticas de las secciones 2 a 6. La validación del
**pipeline real del proyecto** está en la sección 13.

In [19]:
# --- CASO NORMAL: el flujo completo sobre el conjunto real ---
pipe = Pipeline([EliminadorColumnas(COLUMNA_ID), ImputadorMediana(COLUMNA_IMPUTAR)])
salida = pipe.ajustar(datos).transformar(datos)

assert len(salida) == len(datos), "El pipeline perdió o duplicó filas"
assert salida[COLUMNA_IMPUTAR].isna().sum() == 0, f"Quedaron nulos en {COLUMNA_IMPUTAR}"
assert COLUMNA_ID not in salida.columns, "El identificador no se eliminó"
print("Caso normal: las tres comprobaciones pasaron.")

Caso normal: las tres comprobaciones pasaron.


In [20]:
# --- CASOS LÍMITE: situaciones extremas pero válidas ---

# 1) Una columna sin ningún nulo no debe alterarse
sin_nulos = pd.DataFrame({COLUMNA_IMPUTAR: [2.0, 5.0, 8.0]})
r1 = ImputadorMediana(COLUMNA_IMPUTAR).ajustar_transformar(sin_nulos)
assert r1[COLUMNA_IMPUTAR].tolist() == [2.0, 5.0, 8.0]
print("Límite 1: columna sin nulos, sin cambios.")

# 2) Una sola categoría genera una sola columna
una_cat = pd.DataFrame({"raceeth_cod": [5.0, 5.0, 5.0]})
r2 = CodificadorNominal("raceeth_cod").ajustar_transformar(una_cat)
print("Límite 2: una categoría ->", list(r2.columns))

# 3) Varianza cero no debe producir división por cero
constante = pd.DataFrame({COLUMNA_ESCALAR: [4.0, 4.0, 4.0]})
r3 = EscaladorEstandar(COLUMNA_ESCALAR).ajustar_transformar(constante)
assert r3[COLUMNA_ESCALAR].notna().all()
print("Límite 3: varianza cero ->", r3[COLUMNA_ESCALAR].tolist())

# 4) Una categoría nueva en prueba no debe crear columna
pipe_cat = Pipeline([CodificadorNominal("sexo_cod")]).ajustar(entrenamiento)
nuevos = prueba.copy()
nuevos.iloc[0, nuevos.columns.get_loc("sexo_cod")] = 9.0      # código que no existe en el codebook
r4 = pipe_cat.transformar(nuevos)
print("Límite 4: categoría nueva no crea columna ->",
      "sexo_cod_9.0" not in r4.columns)


Límite 1: columna sin nulos, sin cambios.
Límite 2: una categoría -> ['raceeth_cod_5.0']
Límite 3: varianza cero -> [0.0, 0.0, 0.0]
Límite 4: categoría nueva no crea columna -> True


In [21]:
# --- EXCEPCIONES: entradas que deben fallar con mensaje claro ---

pruebas = [
    ("transformar sin ajustar", lambda: ImputadorMediana(COLUMNA_IMPUTAR).transformar(datos)),
    ("columna inexistente", lambda: ImputadorMediana("no_existe").ajustar(datos)),
    ("agregar algo que no es Transformador", lambda: Pipeline().agregar("texto")),
    ("pipeline sin ajustar", lambda: Pipeline([ImputadorMediana(COLUMNA_IMPUTAR)]).transformar(datos)),
]

for descripcion, accion in pruebas:
    try:
        accion()
        print(f"  {descripcion:<38} NO lanzó excepción (revisar)")
    except (RuntimeError, KeyError, TypeError) as error:
        print(f"  {descripcion:<38} {type(error).__name__} capturado")

  transformar sin ajustar                RuntimeError capturado
  columna inexistente                    KeyError capturado
  agregar algo que no es Transformador   TypeError capturado
  pipeline sin ajustar                   RuntimeError capturado


---
# 8. Recursividad

Toda función recursiva tiene dos partes: un **caso base** que la detiene y un **caso
recursivo** que reduce el problema y vuelve a llamarse.

**Cuándo se justifica:** cuando no se sabe de antemano cuán profundo es el problema.
Si la profundidad es fija, un bucle es más simple y más rápido.

In [22]:
def aplanar(estructura, prefijo=""):
    """Convierte metadatos anidados en pares plano de clave y valor.

    Por qué recursión y no un bucle: la profundidad no se conoce al escribir el
    código. Mañana alguien agrega un nivel y el bucle anidado deja de servir.
    """
    plano = {}
    for clave in estructura:
        valor = estructura[clave]
        compuesta = f"{prefijo}.{clave}" if prefijo else str(clave)
        if isinstance(valor, dict) and valor:
            plano.update(aplanar(valor, compuesta))
        else:
            plano[compuesta] = valor
    return plano


metadatos = {
    "proyecto": {"nombre": "YRBS 2023 - redes sociales y salud mental", "fase": 3},
    "datos": {"filas": len(datos), "nulos": {COLUMNA_IMPUTAR: int(datos[COLUMNA_IMPUTAR].isna().sum())}},
    "entorno": {"semilla": SEMILLA, "librerias": {"pandas": pd.__version__}},
}

for clave, valor in aplanar(metadatos).items():
    print(f"{clave:<26} {valor}")

proyecto.nombre            YRBS 2023 - redes sociales y salud mental
proyecto.fase              3
datos.filas                20103
datos.nulos.actividad_fisica_cod 1227
entorno.semilla            42
entorno.librerias.pandas   3.0.6


**Decisión del proyecto sobre recursividad.** En nuestro pipeline la recursión se justifica en
un solo lugar: `aplanar()`, porque los metadatos del proyecto son un diccionario anidado cuya
profundidad puede crecer. Los pasos de transformación, en cambio, recorren una lista fija de
columnas, y para eso un bucle o una operación vectorizada es más simple y más rápido.

En la Formativa 3 se midió además que una recursión que avanza de una fila en una fila falla
por límite de profundidad sobre ~1.000 filas, mientras que la versión *divide y vencerás*
funciona pero es la más lenta. Por eso la regla que adoptamos es: **recursión solo donde la
profundidad es desconocida; división funcional en todo lo demás.**

---
# 9. Eficiencia: medir tiempo y memoria

La rúbrica pide mediciones **reproducibles** con `timeit` o equivalente, comparación
entre implementaciones e interpretación considerando tiempo **y** memoria.

In [23]:
def con_bucle(df):
    """Clasifica las horas de sueño recorriendo las filas una por una.

    Códigos de q85: 1 = ≤4 h, 2 = 5 h, 3 = 6 h, 4 = 7 h, 5 = 8 h, 6 = 9 h, 7 = ≥10 h.
    """
    categorias = []
    for valor in df[COLUMNA_CLASIFICAR]:
        if pd.isna(valor):
            categorias.append("sin dato")
        elif valor < 3:
            categorias.append("muy insuficiente")   # ≤5 h
        elif valor < 5:
            categorias.append("insuficiente")       # 6-7 h
        else:
            categorias.append("suficiente")         # ≥8 h
    return categorias


def vectorizada(df):
    """Lo mismo, con una operación sobre la columna completa."""
    return pd.cut(df[COLUMNA_CLASIFICAR], bins=[-np.inf, 3, 5, np.inf],
                  labels=["muy insuficiente", "insuficiente", "suficiente"],
                  right=False).astype(object).fillna("sin dato").tolist()


# Antes de comparar: comprobar que dan el mismo resultado
assert con_bucle(datos) == vectorizada(datos), "Las versiones no coinciden"
print("Las dos implementaciones producen el mismo resultado.\n")

t_bucle = timeit.timeit(lambda: con_bucle(datos), number=20)
t_vect = timeit.timeit(lambda: vectorizada(datos), number=20)

print(f"Con bucle   : {t_bucle:.4f} s")
print(f"Vectorizada : {t_vect:.4f} s")
print(f"La vectorizada es {t_bucle / t_vect:.1f} veces más rápida")


Las dos implementaciones producen el mismo resultado.

Con bucle   : 0.2405 s
Vectorizada : 0.0613 s
La vectorizada es 3.9 veces más rápida


**La comprobación con `assert` es obligatoria.** Una versión más rápida que entrega otro
resultado no es una optimización: es un error. Decláren­la en el informe.

In [24]:
def medir(funcion, *args, **kwargs):
    """Ejecuta la función y devuelve resultado, segundos y memoria pico en MB.

    *args recoge los argumentos posicionales en una tupla y **kwargs los
    argumentos con nombre en un diccionario. El asterisco es lo que hace el
    trabajo; los nombres args y kwargs son solo convención.
    """
    tracemalloc.start()
    inicio = time.perf_counter()
    resultado = funcion(*args, **kwargs)
    transcurrido = time.perf_counter() - inicio
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return resultado, transcurrido, pico / 1024 / 1024


_, s_bucle, m_bucle = medir(con_bucle, datos)
_, s_vect, m_vect = medir(vectorizada, datos)

pd.DataFrame([
    {"version": "con bucle", "segundos": round(s_bucle, 5), "memoria_mb": round(m_bucle, 3)},
    {"version": "vectorizada", "segundos": round(s_vect, 5), "memoria_mb": round(m_vect, 3)},
])

,version,segundos,memoria_mb
0,con bucle,0.01463,0.166
1,vectorizada,0.00766,0.656


**Atención al contraste.** A veces la versión más rápida consume **más** memoria. Esa
tensión entre tiempo y espacio es exactamente lo que el descriptor pide analizar.

In [25]:
# Cómo crece el costo con el tamaño: esto es lo que revela la complejidad
mediciones = []
for n in [1000, 5000, 20000, 50000]:
    muestra = datos.sample(n=n, replace=True, random_state=SEMILLA)
    t_b = timeit.timeit(lambda: con_bucle(muestra), number=5) / 5
    t_v = timeit.timeit(lambda: vectorizada(muestra), number=5) / 5
    mediciones.append({"filas": n, "bucle_ms": round(t_b * 1000, 2),
                       "vectorizada_ms": round(t_v * 1000, 2),
                       "razon": round(t_b / t_v, 1)})

tabla = pd.DataFrame(mediciones)
print(tabla.to_string(index=False))
print("\nLa última columna es lo que importa: la ventaja crece con el tamaño.")

 filas  bucle_ms  vectorizada_ms  razon
  1000      1.35            1.97    0.7
  5000      3.40            2.69    1.3
 20000     11.99            2.89    4.1
 50000     29.69            5.89    5.0

La última columna es lo que importa: la ventaja crece con el tamaño.


**Un resultado que conviene mirar con atención.** En los tamaños más pequeños la
versión vectorizada puede resultar **igual o más lenta**. No es un error: `pd.cut`
tiene un costo fijo de preparación que en mil filas pesa más que el ahorro. La ventaja
aparece cuando el volumen crece y ese costo fijo se reparte entre más datos.

Es exactamente el tipo de observación que la rúbrica premia: no basta con reportar que
una versión ganó, hay que explicar **bajo qué condiciones**.

**Tres reglas para que la medición valga**

1. **Repitan y conserven el tiempo menor**, no el promedio: los valores altos suelen
   reflejar interrupciones del sistema operativo, no el costo del código.
2. **Midan sobre el tamaño real** de su conjunto. En mil filas todo es instantáneo.
3. **Comprueben que las versiones dan el mismo resultado.**

**Lo que distingue el nivel superior:** que la medición **cambie una decisión**. Medir y
no cambiar nada es un ejercicio; medir, ver dónde está el costo y reescribir esa parte
es optimización.

---
# 10. Patrones de diseño

Un patrón es una solución conocida a un problema que se repite. En esta sección se muestran
los tres que tienen sentido en nuestro proyecto. **El que adoptamos como patrón principal es
Strategy**, aplicado al tratamiento de faltantes (sección 14). Factory y Observer se usan
como apoyo.

## Strategy: varias formas de hacer lo mismo

En la Fase 2 comparamos cuatro formas de tratar los faltantes (conservar, eliminar filas,
moda y mediana) escribiendo un bloque distinto para cada una. Si cada forma es un bloque
distinto, cambiar de una a otra obliga a reescribir. Strategy separa el **qué** del **cómo**.

El ejemplo de esta celda es didáctico (imputación de `actividad_fisica_cod`). La versión del
proyecto está en la sección 14.

In [26]:
class EstrategiaImputacion:
    """Contrato común de todas las estrategias."""
    etiqueta = "sin definir"

    def calcular(self, df, columna):
        raise NotImplementedError


class PorMediana(EstrategiaImputacion):
    etiqueta = "mediana"

    def calcular(self, df, columna):
        return float(df[columna].median())


class PorMedia(EstrategiaImputacion):
    etiqueta = "media"

    def calcular(self, df, columna):
        return float(df[columna].mean())


class PorMedianaDeGrupo(EstrategiaImputacion):
    etiqueta = "mediana por grupo"

    def __init__(self, columna_grupo):
        self.columna_grupo = columna_grupo

    def calcular(self, df, columna):
        return df.groupby(self.columna_grupo)[columna].median().to_dict()


class ImputadorFlexible(Transformador):
    """No sabe imputar: sabe cuándo. El cómo lo aporta la estrategia."""

    def __init__(self, columna, estrategia=None):
        super().__init__(columna)          # llama al constructor de la clase base
        self.estrategia = estrategia or PorMediana()

    def aprender(self, df):
        return {"valor": self.estrategia.calcular(df, self.columna),
                "estrategia": self.estrategia.etiqueta}

    def aplicar(self, df):
        valor = self._parametros["valor"]
        if isinstance(valor, dict):                       # mediana por grupo
            relleno = df[self.estrategia.columna_grupo].map(valor)
            df[self.columna] = df[self.columna].fillna(relleno)
            df[self.columna] = df[self.columna].fillna(df[self.columna].median())
        else:
            df[self.columna] = df[self.columna].fillna(valor)
        return df


# La comparación de estrategias que pide la rúbrica, en un bucle
desv_original = datos[COLUMNA_IMPUTAR].std()
comparacion = []
for estrategia in [PorMediana(), PorMedia(), PorMedianaDeGrupo(COLUMNA_GRUPO)]:
    salida = ImputadorFlexible(COLUMNA_IMPUTAR, estrategia).ajustar_transformar(datos)
    comparacion.append({
        "estrategia": estrategia.etiqueta,
        "desv_antes": round(desv_original, 4),
        "desv_despues": round(salida[COLUMNA_IMPUTAR].std(), 4),
        "cambio_pct": round((salida[COLUMNA_IMPUTAR].std() - desv_original) / desv_original * 100, 2),
    })

pd.DataFrame(comparacion)

,estrategia,desv_antes,desv_despues,cambio_pct
0,mediana,2.503,2.4254,-3.10
1,media,2.503,2.4254,-3.10
2,mediana por grupo,2.503,2.4319,-2.84


**Nota sobre estos números (YRBS).** `actividad_fisica_cod` es un código discreto de 1 a 8, por lo que
la mediana y la media no coinciden y cada estrategia reduce la dispersión en distinta medida. Es solo una
comparación didáctica: en el proyecto **no se imputa** esta variable (bitácora 2.5).

Esa tabla es la comparación que el criterio de preprocesamiento pide justificar. Toda
imputación por un valor central reduce la dispersión; lo que se compara es **cuánto**, y
esa cifra es el argumento para elegir.

## Factory: decidir qué construir

La transformación que corresponde depende del rol de la variable. Factory concentra esa
decisión en un solo lugar, en vez de repetirla en cada cuaderno.

In [27]:
def crear_transformador(rol, columna, **parametros):
    """Devuelve el transformador que corresponde al rol analítico."""
    rol = rol.strip().lower()

    if rol == "continua":
        return ImputadorFlexible(columna, parametros.get("estrategia"))
    if rol == "nominal":
        return CodificadorNominal(columna)
    if rol == "escalar":
        return EscaladorEstandar(columna)
    if rol == "identificador":
        return EliminadorColumnas(columna)

    raise ValueError(
        f"Rol desconocido: '{rol}'. "
        "Roles válidos: continua, nominal, escalar, identificador."
    )


# El diccionario de variables de la Fase 1 pasa a gobernar el pipeline
diccionario = [
    {"variable": COLUMNA_ID, "rol": "identificador"},
    {"variable": COLUMNA_IMPUTAR, "rol": "continua"},
    {"variable": "raceeth_cod", "rol": "nominal"},
    {"variable": "sexo_cod", "rol": "nominal"},
    {"variable": COLUMNA_ESCALAR, "rol": "escalar"},
]

pipeline_auto = Pipeline([crear_transformador(v["rol"], v["variable"]) for v in diccionario])
salida = pipeline_auto.ajustar(datos).transformar(datos)

print(pipeline_auto.resumen().to_string(index=False))
print("\nResultado:", salida.shape)

try:
    crear_transformador("geografica", "estrato")
except ValueError as error:
    print("\nValueError:", error)

 orden                                    paso              clase
     1         EliminadorColumnas(id_registro) EliminadorColumnas
     2 ImputadorFlexible(actividad_fisica_cod)  ImputadorFlexible
     3         CodificadorNominal(raceeth_cod) CodificadorNominal
     4            CodificadorNominal(sexo_cod) CodificadorNominal
     5             EscaladorEstandar(edad_cod)  EscaladorEstandar

Resultado: (20103, 33)

ValueError: Rol desconocido: 'geografica'. Roles válidos: continua, nominal, escalar, identificador.


## Observer: registrar sin estorbar

La bitácora de decisiones suele escribirse a mano después de ejecutar, con el riesgo de
quedar desactualizada. Con Observer, el pipeline **avisa** cada vez que termina un paso
y quien quiera registrar se suscribe.

In [28]:
class Bitacora:
    """Observador que acumula lo que ocurre y lo entrega como tabla."""

    def __init__(self):
        self.registros = []

    def notificar(self, paso, filas, columnas, segundos, parametros):
        self.registros.append({"paso": paso, "filas": filas, "columnas": columnas,
                               "segundos": round(segundos, 5), "parametros": parametros})

    def a_dataframe(self):
        return pd.DataFrame(self.registros)


class ReporteConsola:
    """Otro observador: imprime mientras ocurre."""

    def notificar(self, paso, filas, columnas, segundos, parametros):
        print(f"  {paso:<34} {filas:>6} filas, {columnas:>3} col  [{segundos:.4f}s]")


class PipelineObservable(Pipeline):
    """Hereda todo de Pipeline y agrega la capacidad de ser observado."""

    def __init__(self, pasos=None):
        super().__init__(pasos)
        self._observadores = []

    def suscribir(self, observador):
        self._observadores.append(observador)
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError("Pipeline: hay que ajustar antes de transformar.")
        resultado = df.copy()
        for paso in self._pasos:
            inicio = time.perf_counter()
            resultado = paso.transformar(resultado)
            transcurrido = time.perf_counter() - inicio
            for observador in self._observadores:
                observador.notificar(paso.nombre, len(resultado), resultado.shape[1],
                                     transcurrido, paso.parametros)
        return resultado


bitacora = Bitacora()
pipe = PipelineObservable([
    EliminadorColumnas(COLUMNA_ID),
    ImputadorFlexible(COLUMNA_IMPUTAR),
    CodificadorNominal("raceeth_cod"),
    EscaladorEstandar(COLUMNA_ESCALAR),
])
pipe.suscribir(bitacora).suscribir(ReporteConsola())

print("Ejecución del pipeline:")
salida = pipe.ajustar(datos).transformar(datos)

print("\nBitácora producida por la propia ejecución:")
bitacora.a_dataframe()[["paso", "filas", "columnas", "segundos"]]

Ejecución del pipeline:
  EliminadorColumnas(id_registro)     20103 filas,  25 col  [0.0038s]
  ImputadorFlexible(actividad_fisica_cod)  20103 filas,  25 col  [0.0044s]
  CodificadorNominal(raceeth_cod)     20103 filas,  32 col  [0.0242s]
  EscaladorEstandar(edad_cod)         20103 filas,  32 col  [0.0103s]

Bitácora producida por la propia ejecución:


,paso,filas,columnas,segundos
0,EliminadorColumnas(id_registro),20103,25,0.00383
1,ImputadorFlexible(actividad_fisica_cod),20103,25,0.00436
2,CodificadorNominal(raceeth_cod),20103,32,0.02419
3,EscaladorEstandar(edad_cod),20103,32,0.01031


**Fíjense en lo que acaba de pasar.** La bitácora no se escribió a mano: se genera cada
vez que el pipeline corre, así que no puede quedar desactualizada.

## Resumen: qué patrón usamos y por qué

| Patrón | El problema que resuelve | Dónde se usa en el proyecto |
|---|---|---|
| **Strategy** (principal) | Varias formas de tratar los faltantes | `TratamientoFaltantes` + `ConservarFaltantes` (sección 14) |
| **Factory** (apoyo) | Armar el pipeline desde las constantes declaradas | `construir_pipeline_proyecto()` (sección 11) |
| **Observer** (apoyo) | Registrar cada paso sin que el pipeline sepa quién anota | `PipelineObservable` + `Bitacora` (secciones 11 y 15) |

**Por qué no Singleton.** La configuración del proyecto ya está en una sola celda y en las
constantes de `src/procesamiento.py`. Un Singleton agregaría estado global, que complica las
pruebas, sin resolver un problema que tengamos.

---
# 11. El núcleo algorítmico: el pipeline de la Fase 2 en clases

Hasta acá los ejemplos fueron didácticos. Esta sección toma **cada paso real** del notebook de
la Fase 2 y lo convierte en una clase hija de `Transformador`. Las constantes (columnas,
escalas del codebook) no se copian: se importan de `src/procesamiento.py`, el mismo módulo que
usó la Fase 2.

| Paso en la Fase 2 (función o celda) | Clase en la Fase 3 | Decisión (bitácora) |
|---|---|---|
| Eliminar `orig_rec` tras comprobar 100 % de nulos | `EliminadorColumnaVacia` | 2.1 |
| `proc.seleccionar_columnas()` | `SeleccionadorColumnas` | 2.3 |
| Bandera `n_faltantes_analisis` / `caso_completo` | `MarcadorFaltantes` | 2.5 |
| No imputar ni eliminar filas | `TratamientoFaltantes` + `ConservarFaltantes` (Strategy) | 2.5 |
| `castear_codigos()` | `CastearCodigos` | 2.4 |
| `proc.transformar_datos()` | `ConversorOrdinal` | 2.4 |
| `codificar_one_hot()` | `CodificadorOneHot` | 2.7 |
| `crear_indicador()` | `IndicadorBinario` (ejercicio 1) | 2.7 |

Todas heredan el control de estado de `Transformador` y solo implementan `aprender()` y
`aplicar()`. Las que trabajan con varias columnas redefinen `columnas_requeridas()`.

In [29]:
class EliminadorColumnaVacia(Transformador):
    """Elimina una columna solo si está 100 % vacía (bitácora 2.1).

    La comprobación se hace en aprender(): si la columna tiene aunque sea un
    dato, el paso se niega a borrarla y avisa. Así un cambio en el archivo
    original no pasa inadvertido.
    """

    def aprender(self, df):
        pct = float(df[self.columna].isna().mean() * 100)
        if pct < 100:
            raise ValueError(f"{self.nombre}: la columna tiene datos "
                             f"({pct:.1f} % nulos); no se elimina.")
        return {"pct_nulos": pct}

    def aplicar(self, df):
        return df.drop(columns=[self.columna])


class SeleccionadorColumnas(Transformador):
    """Selecciona y renombra las columnas de análisis (bitácora 2.3)."""

    def __init__(self, mapa):
        super().__init__("columnas de análisis")    # constructor de la base
        self.mapa = dict(mapa)                      # original -> nombre del proyecto

    def columnas_requeridas(self):                  # redefine el de la base
        return list(self.mapa)

    def aprender(self, df):
        # El identificador debe ser único antes de confiar en él
        if not df["record"].is_unique:
            raise ValueError(f"{self.nombre}: 'record' tiene valores repetidos.")
        return {"columnas_entrada": df.shape[1], "columnas_salida": len(self.mapa)}

    def aplicar(self, df):
        return df[list(self.mapa)].rename(columns=self.mapa)


class MarcadorFaltantes(Transformador):
    """Agrega la bandera de faltantes: cuántas variables faltan y si el caso está completo."""

    def __init__(self, columnas):
        super().__init__("bandera de faltantes")
        self.columnas = list(columnas)

    def columnas_requeridas(self):
        return self.columnas

    def aprender(self, df):
        completos = int(df[self.columnas].notna().all(axis=1).sum())
        return {"variables": len(self.columnas), "casos_completos": completos}

    def aplicar(self, df):
        # Versión vectorizada: es la que ganó la comparación de la sección 12
        df["n_faltantes_analisis"] = df[self.columnas].isna().sum(axis=1).astype("int8")
        df["caso_completo"] = (df["n_faltantes_analisis"] == 0).astype("int8")
        return df

In [30]:
class CastearCodigos(Transformador):
    """Convierte códigos float64 a entero Int64, conservando los faltantes (bitácora 2.4)."""

    def __init__(self, columnas):
        super().__init__("códigos a Int64")
        self.columnas = list(columnas)

    def columnas_requeridas(self):
        return self.columnas

    def aprender(self, df):
        # Un código 2.5 no existe en el codebook: se detiene antes de convertir
        for columna in self.columnas:
            valores = df[columna].dropna()
            no_enteros = valores[valores != np.floor(valores)]
            if len(no_enteros):
                raise ValueError(f"{self.nombre}: '{columna}' tiene valores no enteros, "
                                 f"p. ej. {no_enteros.iloc[0]}")
        return {"columnas": len(self.columnas)}

    def aplicar(self, df):
        for columna in self.columnas:
            df[columna] = df[columna].astype("Int64")
        return df


class ConversorOrdinal(Transformador):
    """Declara el orden de las escalas ordinales con el orden del codebook."""

    def __init__(self, escalas):
        super().__init__("escalas ordinales")
        self.escalas = {c: list(v) for c, v in escalas.items()}

    def columnas_requeridas(self):
        return list(self.escalas)

    def aprender(self, df):
        return {"escalas": {c: len(v) for c, v in self.escalas.items()}}

    def aplicar(self, df):
        for columna, categorias in self.escalas.items():
            df[columna] = pd.Categorical(df[columna], categories=categorias, ordered=True)
        return df


class CodificadorOneHot(Transformador):
    """Codifica una nominal en columnas 0/1 con nombres declarados; <NA> donde falta.

    A diferencia del CodificadorNominal didáctico, este NO borra la columna
    original (se conserva para trazabilidad) y NO convierte un faltante en 0.
    """

    def __init__(self, columna, nombres):
        super().__init__(columna)
        self.nombres = dict(nombres)          # código -> nombre de la columna nueva

    def aprender(self, df):
        observados = set(df[self.columna].dropna().astype(int).unique())
        no_declarados = observados - set(self.nombres)
        if no_declarados:
            raise ValueError(f"{self.nombre}: códigos no declarados "
                             f"{sorted(int(c) for c in no_declarados)}")
        return {"columnas_nuevas": list(self.nombres.values())}

    def aplicar(self, df):
        falta = df[self.columna].isna()
        for codigo, nombre in self.nombres.items():
            nueva = (df[self.columna] == codigo).astype("Int8")
            nueva[falta] = pd.NA
            df[nombre] = nueva
        return df

## Ejercicio 1 · Encapsular un paso de nuestro pipeline: `IndicadorBinario`

En la Fase 2, `crear_indicador()` era una función suelta que se llamaba cinco veces. Ahora es
una clase hija de `Transformador`:

- **Herencia:** recibe de la base el control de estado, la validación de columnas y la copia defensiva.
- **Encapsulamiento:** los valores positivos y el porcentaje calculado quedan en `_parametros`, de solo lectura.
- **Regla del proyecto que se mantiene:** un faltante sigue siendo `<NA>`, nunca se convierte en 0.

La prueba de que funciona: el indicador que produce la clase debe ser igual a la columna
`salud_mental_mala` que dejó la Fase 2.

In [31]:
class IndicadorBinario(Transformador):
    """Crea una columna 0/1 que vale 1 si el código está en los valores positivos.

    Conserva <NA> donde el original falta: no convierte un faltante en 0.
    """

    def __init__(self, columna, valores_positivos, nuevo_nombre):
        super().__init__(columna)
        # Se valida al construir: un indicador sin valores positivos no tiene sentido
        if not valores_positivos:
            raise ValueError("valores_positivos no puede estar vacío.")
        self.valores_positivos = list(valores_positivos)
        self.nuevo_nombre = nuevo_nombre

    def _codigos(self, df):
        # Método interno (guion bajo): convierte a número aunque la columna sea Categorical
        return pd.to_numeric(df[self.columna].astype("object"), errors="coerce")

    def aprender(self, df):
        codigos = self._codigos(df)
        con_dato = codigos.notna()
        pct = float(codigos[con_dato].isin(self.valores_positivos).mean() * 100)
        return {"positivos": self.valores_positivos, "pct_positivos": round(pct, 1)}

    def aplicar(self, df):
        codigos = self._codigos(df)
        indicador = codigos.isin(self.valores_positivos).astype("Int8")
        indicador[codigos.isna()] = pd.NA
        df[self.nuevo_nombre] = indicador
        return df


# Prueba con nuestros datos: salud mental no buena = códigos 4 y 5 (Most of the time, Always)
paso = IndicadorBinario("salud_mental_cod", [4, 5], "salud_mental_mala_f3")
salida = paso.ajustar_transformar(datos)

print(paso.nombre, "->", paso.parametros)
print("Nulos en el origen     :", int(datos["salud_mental_cod"].isna().sum()))
print("Nulos en el indicador  :", int(salida["salud_mental_mala_f3"].isna().sum()), "(se conservan)")

coincide = salida["salud_mental_mala_f3"].astype("float").equals(datos["salud_mental_mala"].astype("float"))
print("¿Igual a la columna de la Fase 2?", coincide)
assert coincide, "El indicador de la clase no coincide con el de la Fase 2"

IndicadorBinario(salud_mental_cod) -> {'positivos': [4, 5], 'pct_positivos': 30.6}
Nulos en el origen     : 4398
Nulos en el indicador  : 4398 (se conservan)
¿Igual a la columna de la Fase 2? True


## Strategy aplicado al proyecto: el tratamiento de faltantes

Antes de armar el pipeline completo se definen las estrategias de faltantes, porque el pipeline
las usa. La justificación del patrón y la comparación entre estrategias están en la sección 14.

La estrategia del proyecto es `ConservarFaltantes`: no cambia ningún valor. Aun así se declara
como un paso explícito, para que la decisión de la bitácora 2.5 quede **dentro del código** y no
solo en un documento.

In [32]:
class EstrategiaFaltantes:
    """Contrato común: recibe una columna y devuelve la columna tratada."""
    etiqueta = "sin definir"

    def tratar(self, serie):
        raise NotImplementedError("Cada estrategia debe implementar tratar().")


class ConservarFaltantes(EstrategiaFaltantes):
    """Estrategia del proyecto (bitácora 2.5): los faltantes se conservan como NA."""
    etiqueta = "conservar NA"

    def tratar(self, serie):
        return serie


class RellenarModa(EstrategiaFaltantes):
    """Alternativa evaluada en F2: rellenar con la categoría más frecuente."""
    etiqueta = "moda"

    def tratar(self, serie):
        return serie.fillna(serie.mode().iloc[0])


class RellenarMediana(EstrategiaFaltantes):
    """Alternativa evaluada en F2: rellenar con la mediana del código."""
    etiqueta = "mediana"

    def tratar(self, serie):
        return serie.fillna(serie.median())


class TratamientoFaltantes(Transformador):
    """No sabe tratar faltantes: sabe cuándo hacerlo. El cómo lo aporta la estrategia."""

    def __init__(self, columnas, estrategia=None):
        super().__init__("faltantes")
        self.columnas = list(columnas)
        self.estrategia = estrategia or ConservarFaltantes()

    def columnas_requeridas(self):
        return self.columnas

    def aprender(self, df):
        return {"estrategia": self.estrategia.etiqueta,
                "nulos_antes": int(df[self.columnas].isna().sum().sum())}

    def aplicar(self, df):
        for columna in self.columnas:
            df[columna] = self.estrategia.tratar(df[columna])   # polimorfismo
        return df


print("Estrategias disponibles:",
      [c.etiqueta for c in EstrategiaFaltantes.__subclasses__()])

Estrategias disponibles: ['conservar NA', 'moda', 'mediana']


## Ejercicio 2 · Armar el pipeline completo del proyecto

`construir_pipeline_proyecto()` es una **fábrica** (Factory): arma los pasos a partir de las
constantes declaradas en `src/procesamiento.py` y en esta celda. El orden es el mismo de la
Fase 2, y cada paso tiene su razón:

1. eliminar `orig_rec`, que está vacía;
2. seleccionar y renombrar las 11 columnas;
3. marcar los faltantes **antes** de cualquier cambio de tipo, sobre los valores originales;
4. tratar los faltantes con la estrategia elegida (conservarlos);
5. convertir los códigos a entero y declarar el orden de las ordinales;
6. crear las columnas derivadas: raza/etnicidad 0/1 y los cinco indicadores.

**Por qué aquí no se separa entrenamiento y prueba.** El proyecto es exploratorio: busca una
asociación, no entrena un modelo. Además, ningún paso aprende un valor de los datos que luego
cambie otros datos (solo validan o usan valores declarados del codebook), así que no hay riesgo
de fuga de datos. Por eso el pipeline se ajusta y se aplica sobre el conjunto completo.

In [33]:
# Constantes del paso de codificación, tal como se declararon en la Fase 2
RAZA = {1: "raza_amerindia", 2: "raza_asiatica", 3: "raza_negra",
        4: "raza_hawaiana_pacifico", 5: "raza_blanca", 6: "raza_hispana",
        7: "raza_multiple_hispana", 8: "raza_multiple_no_hispana"}

INDICADORES = [
    ("sexo_cod",             [1],       "sexo_femenino"),
    ("salud_mental_cod",     [4, 5],    "salud_mental_mala"),
    ("redes_sociales_cod",   [6, 7, 8], "redes_uso_frecuente"),
    ("sueno_cod",            [5, 6, 7], "sueno_8h_o_mas"),
    ("actividad_fisica_cod", [6, 7, 8], "actividad_5_dias"),
]

# Las 7 variables de análisis, con su nombre de proyecto
VARIABLES_ANALISIS = [proc.COLUMNAS_ANALISIS[v] for v in proc.COLUMNAS_SIN_DEPENDENCIA]


def construir_pipeline_proyecto(estrategia=None, observable=False):
    """Fábrica: arma el pipeline de la Fase 2 como una lista de Transformadores.

    estrategia : EstrategiaFaltantes, opcional (por defecto ConservarFaltantes)
    observable : si es True, devuelve un PipelineObservable (patrón Observer)
    """
    pasos = [
        EliminadorColumnaVacia("orig_rec"),
        SeleccionadorColumnas(proc.COLUMNAS_ANALISIS),
        MarcadorFaltantes(VARIABLES_ANALISIS),
        TratamientoFaltantes(VARIABLES_ANALISIS, estrategia),
        CastearCodigos(VARIABLES_ANALISIS),
        ConversorOrdinal(proc.ESCALAS_ORDINALES),
        CodificadorOneHot("raceeth_cod", RAZA),
    ]
    pasos += [IndicadorBinario(col, positivos, nombre) for col, positivos, nombre in INDICADORES]

    clase = PipelineObservable if observable else Pipeline
    return clase(pasos)


# El pipeline parte del ARCHIVO ORIGINAL del CDC, igual que la Fase 2
original = proc.cargar_datos(str(RUTA_ORIGINAL))
print(f"Archivo original: {original.shape[0]} filas x {original.shape[1]} columnas\n")

pipeline_proyecto = construir_pipeline_proyecto(observable=True)
bitacora_proyecto = Bitacora()
pipeline_proyecto.suscribir(bitacora_proyecto).suscribir(ReporteConsola())

print(f"Pipeline con {len(pipeline_proyecto)} pasos. Ejecución:")
proyecto = pipeline_proyecto.ajustar(original).transformar(original)
print(f"\nResultado: {proyecto.shape[0]} filas x {proyecto.shape[1]} columnas")

Archivo original: 20103 filas x 117 columnas

Pipeline con 12 pasos. Ejecución:
  EliminadorColumnaVacia(orig_rec)    20103 filas, 116 col  [0.0129s]
  SeleccionadorColumnas(columnas de análisis)  20103 filas,  11 col  [0.0087s]
  MarcadorFaltantes(bandera de faltantes)  20103 filas,  13 col  [0.0051s]
  TratamientoFaltantes(faltantes)     20103 filas,  13 col  [0.0015s]
  CastearCodigos(códigos a Int64)     20103 filas,  13 col  [0.0045s]
  ConversorOrdinal(escalas ordinales)  20103 filas,  13 col  [0.0042s]
  CodificadorOneHot(raceeth_cod)      20103 filas,  21 col  [0.0065s]
  IndicadorBinario(sexo_cod)          20103 filas,  22 col  [0.0066s]
  IndicadorBinario(salud_mental_cod)  20103 filas,  23 col  [0.0071s]
  IndicadorBinario(redes_sociales_cod)  20103 filas,  24 col  [0.0078s]
  IndicadorBinario(sueno_cod)         20103 filas,  25 col  [0.0082s]
  IndicadorBinario(actividad_fisica_cod)  20103 filas,  26 col  [0.0080s]

Resultado: 20103 filas x 26 columnas


In [34]:
# Lo que aprendió cada paso: queda registrado por la propia ejecución
pd.DataFrame([
    {"orden": i, "paso": p.nombre, "parametros": p.parametros}
    for i, p in enumerate(pipeline_proyecto.pasos_ejecutados(), start=1)
])

,orden,paso,parametros
0,1,EliminadorColumnaVacia(orig_rec),{'pct_nulos': 100.0}
1,2,SeleccionadorColumnas(columnas de análisis),"{'columnas_entrada': 116, 'columnas_salida': 11}"
2,3,MarcadorFaltantes(bandera de faltantes),"{'variables': 7, 'casos_completos': 10941}"
3,4,TratamientoFaltantes(faltantes),"{'estrategia': 'conservar NA', 'nulos_antes': ..."
4,5,CastearCodigos(códigos a Int64),{'columnas': 7}
5,6,ConversorOrdinal(escalas ordinales),"{'escalas': {'edad_cod': 7, 'redes_sociales_co..."
6,7,CodificadorOneHot(raceeth_cod),"{'columnas_nuevas': ['raza_amerindia', 'raza_a..."
7,8,IndicadorBinario(sexo_cod),"{'positivos': [1], 'pct_positivos': 49.6}"
8,9,IndicadorBinario(salud_mental_cod),"{'positivos': [4, 5], 'pct_positivos': 30.6}"
9,10,IndicadorBinario(redes_sociales_cod),"{'positivos': [6, 7, 8], 'pct_positivos': 78.1}"


**Qué muestra esta tabla.** Cada fila es un objeto distinto, pero el pipeline los recorrió a
todos con las mismas dos llamadas (`ajustar` y `transformar`) sin preguntar de qué tipo eran.
Eso es **polimorfismo** aplicado a nuestro propio pipeline. Y agregar un paso nuevo (por ejemplo,
un indicador más) no obliga a tocar la clase `Pipeline`: solo se agrega una línea en la fábrica.

---
# 12. Ejercicio 3 · Eficiencia sobre un paso real: contar faltantes por fila

El paso `MarcadorFaltantes` cuenta, para cada uno de los 20.103 estudiantes, cuántas de las 7
variables de análisis están vacías. Hay tres formas de escribirlo:

| Versión | Cómo trabaja |
|---|---|
| `con_bucle` | Recorre las filas una por una con `itertuples()` y cuenta en Python |
| `con_apply` | `DataFrame.apply(..., axis=1)`: parece vectorizado, pero llama a una función de Python por cada fila |
| `vectorizada` | `isna().sum(axis=1)`: una sola operación sobre la tabla completa, ejecutada en C por NumPy |

Las tres tienen la misma complejidad teórica, **O(n·k)** (n filas, k = 7 columnas): todas miran
cada celda una vez. Lo que cambia es el **costo por celda**, y eso es lo que se mide.

In [35]:
seleccion = SeleccionadorColumnas(proc.COLUMNAS_ANALISIS).ajustar_transformar(
    EliminadorColumnaVacia("orig_rec").ajustar_transformar(original))


def con_bucle(df, columnas=VARIABLES_ANALISIS):
    """Cuenta faltantes fila por fila, en Python puro."""
    return [sum(1 for valor in fila if pd.isna(valor))
            for fila in df[columnas].itertuples(index=False)]


def con_apply(df, columnas=VARIABLES_ANALISIS):
    """Cuenta faltantes con apply por fila: una llamada de Python por cada fila."""
    return df[columnas].apply(lambda fila: int(fila.isna().sum()), axis=1).tolist()


def vectorizada(df, columnas=VARIABLES_ANALISIS):
    """Cuenta faltantes con una operación sobre la tabla completa."""
    return df[columnas].isna().sum(axis=1).tolist()


# 1) Antes de medir: las tres versiones deben dar EXACTAMENTE el mismo resultado
assert con_bucle(seleccion) == con_apply(seleccion) == vectorizada(seleccion), \
    "Las versiones no coinciden"
print("Las tres versiones producen el mismo conteo en las", len(seleccion), "filas.\n")

# 2) Tiempo: se repite y se conserva el MENOR (regla de la sección 9)
filas_tiempo = []
for funcion in (con_bucle, con_apply, vectorizada):
    mejor = min(timeit.repeat(lambda: funcion(seleccion), number=1, repeat=5))
    _, _, memoria = medir(funcion, seleccion)
    filas_tiempo.append({"version": funcion.__name__, "ms": round(mejor * 1000, 2),
                         "memoria_pico_mb": round(memoria, 3)})

comparacion_faltantes = pd.DataFrame(filas_tiempo)
comparacion_faltantes["veces_mas_lenta"] = (
    comparacion_faltantes["ms"] / comparacion_faltantes["ms"].min()).round(1)
comparacion_faltantes

Las tres versiones producen el mismo conteo en las 20103 filas.



,version,ms,memoria_pico_mb,veces_mas_lenta
0,con_bucle,77.84,1.250,16.0
1,con_apply,966.14,3.509,198.8
2,vectorizada,4.86,1.211,1.0


In [36]:
# 3) Cómo crece el costo con el tamaño del conjunto
crecimiento = []
for n in [1_000, 5_000, 20_103, 40_000]:
    muestra = seleccion.sample(n=n, replace=n > len(seleccion), random_state=SEMILLA)
    fila = {"filas": n}
    for funcion in (con_bucle, con_apply, vectorizada):
        fila[f"{funcion.__name__}_ms"] = round(
            min(timeit.repeat(lambda: funcion(muestra), number=1, repeat=3)) * 1000, 2)
    crecimiento.append(fila)

tabla_crecimiento = pd.DataFrame(crecimiento)
tabla_crecimiento["apply / vectorizada"] = (
    tabla_crecimiento["con_apply_ms"] / tabla_crecimiento["vectorizada_ms"]).round(0)
print(tabla_crecimiento.to_string(index=False))

 filas  con_bucle_ms  con_apply_ms  vectorizada_ms  apply / vectorizada
  1000         10.36        108.62            2.75                 39.0
  5000         22.80        237.37            1.18                201.0
 20103         75.02       1027.51            4.43                232.0
 40000        147.96       1949.99            5.39                362.0


**Interpretación.** Los números exactos cambian de un computador a otro, pero el patrón se repite:

1. **Las tres versiones crecen de forma lineal**: si se duplican las filas, el tiempo se duplica
   aproximadamente. Eso confirma la complejidad O(n·k) de las tres.
2. **La diferencia está en la constante.** `vectorizada` hace el trabajo en C sobre la tabla
   completa; `con_bucle` lo hace en Python, celda por celda; y `con_apply` es **la más lenta**
   porque además crea un objeto `Series` por cada fila. Es un resultado que sorprende: `apply`
   *parece* vectorizado, pero no lo es.
3. **Memoria.** En nuestra medición la versión vectorizada fue también la de menor pico de
   memoria, y `con_apply` la de mayor, porque crea un objeto por cada fila. No siempre es así:
   la versión vectorizada arma una tabla temporal de verdadero/falso del tamaño de las 7
   columnas, que con conjuntos mucho más grandes podría pesar. Con 20.103 filas esa tabla es
   pequeña y no hay tensión entre tiempo y memoria.

**Decisión.** `MarcadorFaltantes` usa la versión vectorizada. La medición **cambió una decisión**:
descartamos `apply`, que era la forma que habríamos escrito por intuición.

---
# 13. Ejercicio 4 · Validación del pipeline real en tres escenarios

Se prueba el pipeline del proyecto (no las clases didácticas) en:

- **Caso normal:** el archivo completo del CDC.
- **Casos límite:** situaciones extremas pero válidas (una fila, un estudiante sin ninguna respuesta, una sola categoría).
- **Excepciones:** entradas que deben detener el pipeline con un mensaje claro.

In [37]:
# --- CASO NORMAL: el archivo completo ---
salida = proyecto
seleccion_ref = seleccion   # las 11 columnas antes de transformar

assert len(salida) == len(original) == 20103, "El pipeline perdió o duplicó filas"
print("[OK] Filas conservadas:", len(salida))

nulos_antes = seleccion_ref[VARIABLES_ANALISIS].isna().sum()
nulos_despues = salida[VARIABLES_ANALISIS].isna().sum()
assert (nulos_antes == nulos_despues).all(), "Cambiaron los faltantes: se imputó algo"
print("[OK] Faltantes idénticos antes y después en las 7 variables:", int(nulos_despues.sum()))

for columna in ["id_registro", "peso_muestral", "estrato", "psu"]:
    assert salida[columna].equals(seleccion_ref[columna]), f"{columna} fue modificada"
print("[OK] Identificador y diseño muestral intactos")

columnas_raza = list(RAZA.values())
con_dato = salida["raceeth_cod"].notna()
assert (salida.loc[con_dato, columnas_raza].sum(axis=1) == 1).all()
assert salida.loc[~con_dato, columnas_raza].isna().all().all()
print(f"[OK] Raza/etnicidad: suma 1 en {int(con_dato.sum())} filas y <NA> en {int((~con_dato).sum())}")

assert (salida["caso_completo"] == (salida["n_faltantes_analisis"] == 0)).all()
print("[OK] Bandera coherente:", int(salida["caso_completo"].sum()), "casos completos")

[OK] Filas conservadas: 20103
[OK] Faltantes idénticos antes y después en las 7 variables: 13813
[OK] Identificador y diseño muestral intactos
[OK] Raza/etnicidad: suma 1 en 19733 filas y <NA> en 370
[OK] Bandera coherente: 10941 casos completos


In [38]:
# --- CASOS LÍMITE ---

def correr(df):
    """Arma un pipeline nuevo y lo ejecuta sobre df (cada prueba parte de cero)."""
    return construir_pipeline_proyecto().ajustar(df).transformar(df)


# 1) Un conjunto de una sola fila funciona igual que el completo
r1 = correr(original.head(1))
assert r1.shape == (1, 26)
print("Límite 1: una sola fila ->", r1.shape)

# 2) Un estudiante que no respondió ninguna de las 7 preguntas
vacio = original.head(3).copy()
vacio.loc[vacio.index[0], proc.COLUMNAS_SIN_DEPENDENCIA] = np.nan
r2 = correr(vacio)
fila = r2.iloc[0]
assert fila["n_faltantes_analisis"] == 7 and fila["caso_completo"] == 0
assert r2.loc[r2.index[0], columnas_raza].isna().all()
assert r2.loc[r2.index[0], [n for _, _, n in INDICADORES]].isna().all()
print("Límite 2: sin ninguna respuesta -> 7 faltantes, caso incompleto, indicadores <NA> (no 0)")

# 3) Una sola categoría de raza: igual se crean las 8 columnas declaradas
blancos = original[original["raceeth"] == 5].head(50)
r3 = correr(blancos)
assert all(c in r3.columns for c in columnas_raza)
assert r3["raza_blanca"].eq(1).all() and r3["raza_negra"].eq(0).all()
print("Límite 3: una sola categoría -> 8 columnas, raza_blanca = 1 en todas")

# 4) Estrategia intercambiada: con moda ya no quedan faltantes, sin tocar otros pasos
r4 = construir_pipeline_proyecto(RellenarModa()).ajustar(original.head(500)).transformar(original.head(500))
assert r4[VARIABLES_ANALISIS].isna().sum().sum() == 0
print("Límite 4: cambiar la estrategia solo cambia el tratamiento de faltantes")

Límite 1: una sola fila -> (1, 26)
Límite 2: sin ninguna respuesta -> 7 faltantes, caso incompleto, indicadores <NA> (no 0)
Límite 3: una sola categoría -> 8 columnas, raza_blanca = 1 en todas
Límite 4: cambiar la estrategia solo cambia el tratamiento de faltantes


In [39]:
# --- EXCEPCIONES: entradas que deben fallar con un mensaje claro ---

def con_cambio(columna, fila, valor):
    """Copia pequeña del original con un valor alterado."""
    df = original.head(20).copy()
    df[columna] = df[columna].astype("object") if isinstance(valor, str) else df[columna]
    df.loc[df.index[fila], columna] = valor
    return df


pruebas = [
    ("código de raza fuera del codebook (9)", lambda: correr(con_cambio("raceeth", 0, 9)), ValueError),
    ("código no entero (q84 = 2.5)",          lambda: correr(con_cambio("q84", 0, 2.5)), ValueError),
    ("orig_rec con un dato",                  lambda: correr(con_cambio("orig_rec", 0, 1)), ValueError),
    ("archivo sin la columna q85",            lambda: correr(original.head(20).drop(columns=["q85"])), KeyError),
    ("record repetido",                       lambda: correr(con_cambio("record", 1, original["record"].iloc[0])), ValueError),
    ("transformar sin ajustar",               lambda: construir_pipeline_proyecto().transformar(original), RuntimeError),
    ("indicador sin valores positivos",       lambda: IndicadorBinario("sueno_cod", [], "x"), ValueError),
]

resultados = []
for descripcion, accion, esperada in pruebas:
    try:
        accion()
        resultados.append({"prueba": descripcion, "resultado": "NO lanzó excepción", "ok": False})
    except esperada as error:
        resultados.append({"prueba": descripcion, "resultado": f"{type(error).__name__}: {str(error)[:70]}", "ok": True})

tabla_excepciones = pd.DataFrame(resultados)
assert tabla_excepciones["ok"].all(), "Alguna excepción esperada no se lanzó"
tabla_excepciones

,prueba,resultado,ok
0,código de raza fuera del codebook (9),ValueError: CodificadorOneHot(raceeth_cod): có...,True
1,código no entero (q84 = 2.5),ValueError: CastearCodigos(códigos a Int64): '...,True
2,orig_rec con un dato,ValueError: EliminadorColumnaVacia(orig_rec): ...,True
3,archivo sin la columna q85,"KeyError: ""SeleccionadorColumnas(columnas de a...",True
4,record repetido,ValueError: SeleccionadorColumnas(columnas de ...,True
5,transformar sin ajustar,RuntimeError: Pipeline: hay que ajustar antes ...,True
6,indicador sin valores positivos,ValueError: valores_positivos no puede estar v...,True


---
# 14. Ejercicio 5 · Strategy y verificación contra la Fase 2

## Por qué Strategy

- **Qué problema resolvía.** En la Fase 2 comparamos cuatro formas de tratar los faltantes y cada una era un bloque de código distinto; cambiar de decisión obligaba a reescribir el pipeline.
- **Cómo lo resuelve el patrón.** `TratamientoFaltantes` sabe *cuándo* tratar los faltantes y delega *cómo* hacerlo en una estrategia intercambiable (`ConservarFaltantes`, `RellenarModa`, `RellenarMediana`), que se elige al construir el pipeline.
- **Qué habría pasado sin él.** Probar otra estrategia (por ejemplo, si en la Fase 4 se decide imputar) habría significado editar el pipeline y arriesgar romper los demás pasos; ahora es cambiar un argumento.

La celda siguiente repite la comparación de la Fase 2, pero ahora **cambiando solo la estrategia**.

In [40]:
# Distribución de salud mental observada (solo quienes respondieron): la referencia
referencia = seleccion[COLUMNA_OBJETIVO].value_counts(normalize=True).sort_index() * 100

filas_estrategia = []
for estrategia in (ConservarFaltantes(), RellenarModa(), RellenarMediana()):
    salida_e = construir_pipeline_proyecto(estrategia).ajustar(original).transformar(original)
    serie = salida_e[COLUMNA_OBJETIVO].astype("float")
    distribucion = serie.value_counts(normalize=True).sort_index() * 100
    filas_estrategia.append({
        "estrategia": estrategia.etiqueta,
        "respuestas_usadas": int(serie.notna().sum()),
        "respuestas_inventadas": int(serie.notna().sum() - seleccion[COLUMNA_OBJETIVO].notna().sum()),
        "max_desvio_pp": round(float((distribucion - referencia).abs().max()), 1),
    })

pd.DataFrame(filas_estrategia)

,estrategia,respuestas_usadas,respuestas_inventadas,max_desvio_pp
0,conservar NA,15705,0,0.0
1,moda,20103,4398,15.3
2,mediana,20103,4398,15.3


**Cómo se lee.** `max_desvio_pp` mide cuánto se aleja la distribución de salud mental de la que
realmente se observó. Rellenar con la moda o la mediana **inventa** miles de respuestas y las
concentra en una sola categoría, que se infla en varios puntos porcentuales. Por eso el proyecto
mantiene `ConservarFaltantes` (bitácora 2.5): es la única estrategia que no altera lo observado.

## La verificación que más rinde: ¿el pipeline de clases produce lo mismo que la Fase 2?

Se escribe la salida del pipeline de clases en formato CSV (en memoria, sin tocar el disco) y se
compara con el archivo que guardó la Fase 2, de dos formas:

1. `pd.testing.assert_frame_equal`: mismas columnas, mismo orden, mismos tipos y mismos valores.
2. Comparación del texto completo del archivo: debe ser **idéntico carácter por carácter**.

In [41]:
# Salida del pipeline de clases, escrita como CSV en memoria.
# lineterminator="\n": en Windows pandas termina las líneas con \r\n, pero al leer
# el archivo en modo texto Python las convierte a \n. Así ambos textos se comparan igual.
buffer = io.StringIO()
proyecto.to_csv(buffer, index=False, lineterminator="\n")
texto_f3 = buffer.getvalue()
nuevo = pd.read_csv(io.StringIO(texto_f3))

# Archivo guardado por la Fase 2
anterior = pd.read_csv(RUTA_DATOS)

# 1) Misma tabla
pd.testing.assert_frame_equal(anterior, nuevo)
print("assert_frame_equal: las dos tablas son iguales",
      f"({anterior.shape[0]} filas x {anterior.shape[1]} columnas)")

# 2) Mismo archivo, carácter por carácter
with open(RUTA_DATOS, encoding="utf-8") as archivo:
    texto_f2 = archivo.read()
identico = texto_f2 == texto_f3
print("Archivo idéntico carácter por carácter:", identico)
assert identico, "El CSV del pipeline de clases difiere del de la Fase 2"

print("\nLa reorganización en clases no alteró ni un valor del resultado de la Fase 2.")

assert_frame_equal: las dos tablas son iguales (20103 filas x 26 columnas)
Archivo idéntico carácter por carácter: True

La reorganización en clases no alteró ni un valor del resultado de la Fase 2.


---
# 15. El resultado: cómo queda el código y qué se registra

## 15.1 El código, antes y después

**Antes (Fase 2).** El pipeline era una secuencia de celdas que llamaban funciones sueltas; el
orden estaba implícito en la posición de las celdas:

```python
df = proc.cargar_datos(RUTA_RAW)
df = df.drop(columns=["orig_rec"])
df_sel = proc.seleccionar_columnas(df)
df_sel["n_faltantes_analisis"] = df_sel[columnas_analisis].isna().sum(axis=1)
df_t = castear_codigos(df_sel, columnas_analisis)
df_t = proc.transformar_datos(df_t)
df_t = codificar_one_hot(df_t, "raceeth_cod", RAZA)
for columna, positivos, nombre in INDICADORES:
    df_t = crear_indicador(df_t, columna, positivos, nombre)
```

**Después (Fase 3).** El mismo trabajo, con el orden declarado en un solo lugar:

```python
pipeline = construir_pipeline_proyecto(estrategia=ConservarFaltantes())
resultado = pipeline.ajustar(original).transformar(original)
```

El resultado es el mismo (sección 14). Lo que cambió es que ahora cada paso es una pieza con
nombre, que se puede probar, reemplazar y reutilizar sin tocar las demás.

## 15.2 La arquitectura, generada desde el propio código

En vez de escribir a mano la tabla de arquitectura, se produce leyendo las clases que existen.
Así no queda desactualizada.

In [42]:
CLASES_PROYECTO = {"EliminadorColumnaVacia", "SeleccionadorColumnas", "MarcadorFaltantes",
                   "TratamientoFaltantes", "CastearCodigos", "ConversorOrdinal",
                   "CodificadorOneHot", "IndicadorBinario"}


def documentar_arquitectura():
    """Genera la tabla de arquitectura a partir de las clases definidas.

    __subclasses__() devuelve las clases que heredan de una clase. Es la misma
    información que Python usa para resolver la herencia, así que la tabla
    refleja el código real y no lo que creemos que hay.
    """
    def primera_linea(clase):
        return (clase.__doc__ or "sin documentar").strip().split("\n")[0]

    filas = [{"componente": "Transformador", "rol": "clase base",
              "uso": "proyecto y didáctico",
              "responsabilidad": "Define el contrato y controla el estado de cada paso",
              "archivo sugerido": "F3/src/transformadores.py"}]

    for clase in Transformador.__subclasses__():
        filas.append({"componente": clase.__name__, "rol": "clase hija",
                      "uso": "proyecto" if clase.__name__ in CLASES_PROYECTO else "didáctico",
                      "responsabilidad": primera_linea(clase),
                      "archivo sugerido": "F3/src/transformadores.py"})

    for clase in EstrategiaFaltantes.__subclasses__():
        filas.append({"componente": clase.__name__, "rol": "estrategia (Strategy)",
                      "uso": "proyecto", "responsabilidad": primera_linea(clase),
                      "archivo sugerido": "F3/src/estrategias.py"})

    filas += [
        {"componente": "Pipeline", "rol": "orquestador", "uso": "proyecto",
         "responsabilidad": "Encadena los pasos y controla el orden de ajuste",
         "archivo sugerido": "F3/src/pipeline.py"},
        {"componente": "PipelineObservable", "rol": "orquestador (Observer)", "uso": "proyecto",
         "responsabilidad": "Pipeline que además notifica cada paso a sus observadores",
         "archivo sugerido": "F3/src/pipeline.py"},
        {"componente": "Bitacora", "rol": "observador", "uso": "proyecto",
         "responsabilidad": "Registra lo ocurrido en cada paso de la ejecución",
         "archivo sugerido": "F3/src/pipeline.py"},
        {"componente": "construir_pipeline_proyecto", "rol": "fábrica (Factory)", "uso": "proyecto",
         "responsabilidad": "Arma el pipeline desde las constantes declaradas",
         "archivo sugerido": "F3/src/pipeline.py"},
        {"componente": "medir", "rol": "utilidad", "uso": "proyecto",
         "responsabilidad": "Mide tiempo y memoria de cualquier función",
         "archivo sugerido": "F3/src/medicion.py"},
    ]
    return pd.DataFrame(filas)


arquitectura = documentar_arquitectura()
arquitectura[arquitectura["uso"] != "didáctico"]

,componente,rol,uso,responsabilidad,archivo sugerido
0,Transformador,clase base,proyecto y didáctico,Define el contrato y controla el estado de cad...,F3/src/transformadores.py
6,EliminadorColumnaVacia,clase hija,proyecto,Elimina una columna solo si está 100 % vacía (...,F3/src/transformadores.py
7,SeleccionadorColumnas,clase hija,proyecto,Selecciona y renombra las columnas de análisis...,F3/src/transformadores.py
8,MarcadorFaltantes,clase hija,proyecto,Agrega la bandera de faltantes: cuántas variab...,F3/src/transformadores.py
9,CastearCodigos,clase hija,proyecto,"Convierte códigos float64 a entero Int64, cons...",F3/src/transformadores.py
10,ConversorOrdinal,clase hija,proyecto,Declara el orden de las escalas ordinales con ...,F3/src/transformadores.py
11,CodificadorOneHot,clase hija,proyecto,Codifica una nominal en columnas 0/1 con nombr...,F3/src/transformadores.py
12,IndicadorBinario,clase hija,proyecto,Crea una columna 0/1 que vale 1 si el código e...,F3/src/transformadores.py
13,TratamientoFaltantes,clase hija,proyecto,No sabe tratar faltantes: sabe cuándo hacerlo....,F3/src/transformadores.py
14,ConservarFaltantes,estrategia (Strategy),proyecto,Estrategia del proyecto (bitácora 2.5): los fa...,F3/src/estrategias.py


**Esta tabla alimenta el apartado III.b del informe** (documentación de arquitectura). Allí se
explica **por qué** esta división: cada archivo tiene una sola responsabilidad y, para cambiar
el tratamiento de faltantes, solo se cambia la estrategia.

## 15.3 Registro de la ejecución

Se guardan en `F3/resultados/` tres archivos que permiten auditar la ejecución sin volver a
correr el cuaderno. **No se guarda un nuevo conjunto de datos**: el de la Fase 2 sigue siendo el
oficial, y la sección 14 demostró que el pipeline de clases lo reproduce exactamente.

In [43]:
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

registro_parametros = pd.DataFrame([
    {"orden": i, "paso": p.nombre, "clase": type(p).__name__, "parametros": str(p.parametros)}
    for i, p in enumerate(pipeline_proyecto.pasos_ejecutados(), start=1)
])
archivos = {
    "parametros_pipeline_f3.csv": registro_parametros,
    "bitacora_ejecucion_f3.csv": bitacora_proyecto.a_dataframe()[["paso", "filas", "columnas", "segundos"]],
    "arquitectura_f3.csv": arquitectura,
    "eficiencia_faltantes_f3.csv": comparacion_faltantes,
    "eficiencia_crecimiento_f3.csv": tabla_crecimiento,
}
for nombre, tabla in archivos.items():
    ruta = CARPETA_RESULTADOS / nombre
    tabla.to_csv(ruta, index=False)
    print(f"Guardado: F3/resultados/{nombre}  ({len(tabla)} filas)")

# Verificación de ida y vuelta de uno de ellos
releido = pd.read_csv(CARPETA_RESULTADOS / "parametros_pipeline_f3.csv")
assert len(releido) == len(pipeline_proyecto)
print("\nRegistro verificado: se puede volver a leer sin sorpresas.")

Guardado: F3/resultados/parametros_pipeline_f3.csv  (12 filas)
Guardado: F3/resultados/bitacora_ejecucion_f3.csv  (12 filas)
Guardado: F3/resultados/arquitectura_f3.csv  (22 filas)
Guardado: F3/resultados/eficiencia_faltantes_f3.csv  (3 filas)
Guardado: F3/resultados/eficiencia_crecimiento_f3.csv  (4 filas)

Registro verificado: se puede volver a leer sin sorpresas.


---
# Cierre

## Dónde está cada concepto en este cuaderno

| Concepto | Evidencia en el pipeline del proyecto |
|---|---|
| **Encapsulamiento** | `_parametros` y `_ajustado` internos; `parametros` y `nombre` de solo lectura; `_codigos()` como método interno de `IndicadorBinario` |
| **Herencia** | 8 clases del proyecto heredan de `Transformador`; `PipelineObservable` hereda de `Pipeline`; 3 estrategias heredan de `EstrategiaFaltantes` |
| **Polimorfismo** | El `Pipeline` llama `ajustar`/`transformar` sin preguntar el tipo; `TratamientoFaltantes` llama `tratar()` sin saber qué estrategia es; `columnas_requeridas()` redefinido en las hijas |
| **Patrones** | Strategy (principal), Factory y Observer (apoyo) |
| **Recursividad** | `aplanar()` para metadatos anidados (sección 8) |
| **Eficiencia** | Conteo de faltantes: bucle vs. `apply` vs. vectorizado, con `timeit`, `tracemalloc` y tabla de crecimiento (sección 12) |
| **Validación** | Caso normal, 4 límites y 7 excepciones sobre el pipeline real (sección 13) |
| **Verificación contra F2** | `assert_frame_equal` y archivo idéntico carácter por carácter (sección 14) |

## Decisiones que quedan registradas

1. El pipeline de la Fase 2 se reorganiza en clases sin cambiar su resultado.
2. Los faltantes se conservan (estrategia `ConservarFaltantes`), ahora como decisión explícita dentro del código.
3. El conteo de faltantes se hace de forma vectorizada: `apply` resultó la opción más lenta.
4. La recursión se usa solo donde la profundidad es desconocida.

## Bibliografía (APA 7)

Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1994). *Design patterns: Elements of
reusable object-oriented software*. Addison-Wesley.

Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D.,
Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk,
M. H., Brett, M., Haldane, A., del Río, J. F., Wiebe, M., Peterson, P., … Oliphant, T. E.
(2020). Array programming with NumPy. *Nature, 585*(7825), 357–362.
https://doi.org/10.1038/s41586-020-2649-2

McKinney, W. (2022). *Python for data analysis* (3.ª ed.). O'Reilly Media.

Python Software Foundation. (s. f.). *Classes*. https://docs.python.org/3/tutorial/classes.html

Python Software Foundation. (s. f.). *timeit — Measure execution time of small code
snippets*. https://docs.python.org/3/library/timeit.html

The pandas development team. (s. f.). *pandas documentation*. https://pandas.pydata.org/docs/

Universidad Andrés Bello. (2026). *MCDI500 Programación para la Ciencia de Datos: Apunte Fase 3*
[Material docente]. Magíster en Ciencia de Datos e Inteligencia Artificial.

---

*Avance Fase 3 · Semana 2 · MCDI500 · Grupo 8 · Magíster en Ciencia de Datos e Inteligencia
Artificial · Universidad Andrés Bello*